**Cell: 1**

# 10 · End-to-end QA with `claude -p` — from ticket to signed-off merge

**Pattern family:** engineering workflow · **Tool:** Claude Code headless mode (`claude -p`)

Notebook 09 put one engineer through a CI/CD pipeline. This one puts **two seats** through the
same pipeline on a **real application** — the Ranger · Data Catalog · Iceberg governance demo —
and never lets the person who wrote the code be the person who signs off on it.

### 🎯 Purpose

- 👤 **Separate the seat that builds from the seat that verifies.** The developer proves the feature
  works; QA proves it broke nothing else. Different jobs, different evidence — and it is the
  second one that catches the defect here.
- 🔹 **Change the schema the way production changes it: with a migration.** The customer table
  lives in a warehouse before the ticket is opened. The feature is an `ALTER TABLE … ADD COLUMN`
  that is versioned, applied and diffed — and the catalog that governs that table is a *second*
  schema the migration never touches. Two schemas, one change, one of them forgotten.
- 🧪 **Put a real application under test, not a toy script.** Ranger row-filters and column masks,
  three identities, a catalog whose tags are the security boundary. The bug is a governance bug,
  and only governance tests see it.
- 🧪 **Find the defect from a failing test, not from a code read.** `phone` lands in the data, the
  planner and the catalog — and the catalog line carries no `tag="PII"`. Nothing in Python, the
  type system or the existing suite will say so. Step 6 will.
- 🤖 **Use `claude -p` as a pipeline step with a grant, not as a chatbot.** Seven calls, each scoped
  with `--allowedTools` to exactly what that seat may do: `Read,Grep,Glob` to plan, `Read,Write` to
  author tests, `Read,Edit` to fix, nothing at all to write a report.
- 🧪 **Make every gate a command with an exit code.** `grep -c blocker`, `git diff --quiet -- tests/`,
  `pytest -q`. A gate checks the *shape* of a reply, never its length or its tone.
- 🧪 **Write the test plan before the tests, from the diff and the ticket.** The plan is the
  contract; the generated suite is pinned to it by name. Notebook 11 shows what happens when
  that plan is weakened.
- ⚠️ **Grade the tests themselves.** Step 10 plants a mutant and expects red. A suite that stays
  green against a broken mask is not a suite.
- 👤 **Leave an evidence trail a reviewer can open.** Every step writes to `qa_evidence/` — baseline,
  diff, plan, RED run, defect report, GREEN run, mutation, security review — and step 12 attaches
  all of it to the PR.
- 👤 **Keep the approval human, and move the gates into CI.** Step 13 is a person reading the
  evidence; step 14 is the same commands in a workflow file so the next PR gets the same scrutiny.

### 👥 The two seats

| Seat | Icon | Owns |
|---|---|---|
| **Developer** | 👨‍💻 | the ticket, the branch, the feature, the commit, the fix, the PR |
| **QA Lead** | 🧪 | the environment, the test plan, the test cases, every gate, the defect report, the sign-off |

You will play both. **Every step below is tagged with the seat that performs it** — in the
heading, and again in the banner comment inside the cell. When the tag flips, your hat flips.

### 🎯 The rule this notebook exists to teach

> **The developer proves the feature works. QA proves the feature did not break anything else.**
> Those are different jobs, they use different evidence, and the second one is the one that
> catches the defect in this notebook.

### 🧪 The system under test

`/Users/satyak/Ranagr_DataCatalog_Iceberg` — a self-contained Streamlit app where an agent turns
English into SQL, and **Apache Ranger** (simulated faithfully) enforces row-filters and column
masks by role. No cluster needed; the whole governance path runs offline, which is exactly what
makes it testable.

| Identity | Role | Tenant | Entitled to see |
|---|---|---|---|
| `dave` | admin | ALL | raw values, every region |
| `alice` | analyst | US | PII → SHA-256 hash, card redacted, **US rows only** |
| `carol` | support | EU | PII → last-4, card redacted, **EU rows only** |

The app's own design note is the thing QA must hold it to:
**the LLM and the app are NOT the security boundary — the catalog tags are, and Ranger enforces them.**

### 🎟️ The work

**Ticket QA-114** — *"Support needs to call customers back. Add a `phone` column to the customer
profile."* One migration and two one-line edits. One missing word in a metadata file. You will find out
which word from a **failing test**, not from a code read.

### 🔁 The pipeline

| # | Step | Seat | The one command that matters |
|---|---|---|---|
| 0 | Clean room + **warehouse** + baseline | 🧪 QA | `sqlite3 … < migrations/000_*.sql` · `python3 tests/test_flow.py` |
| 1 | Ticket, branch | 👨‍💻 DEV | `git switch -c feature/qa-114-phone` |
| 2 | Build the feature | 👨‍💻 DEV | `ALTER TABLE` migration + two `sed` |
| 3 | Commit message | 👨‍💻 DEV | `claude -p` reads the staged diff |
| 4 | **Test plan** from the diff | 🧪 QA | `claude -p --allowedTools "Read,Grep,Glob"` |
| 5 | Write the test cases | 🧪 QA | `claude -p --allowedTools "Read,Write"` |
| 6 | First run — expect **RED** | 🧪 QA | `pytest -q` → `grep -c failed` |
| 7 | Defect report, with severity | 🧪 QA | `claude -p` → JSON → `grep blocker` |
| 8 | Fix what QA found | 👨‍💻 DEV | `claude -p --allowedTools "Read,Edit"` |
| 9 | Re-run — **GREEN** + regression | 🧪 QA | `pytest -q` **and** the legacy suite |
| 10 | Grade the tests (mutation) | 🧪 QA | `sed` a mutant in, suite must go red |
| 11 | Security review of the final diff | 🧪 QA | `claude -p` → JSON → blockers must be 0 |
| 12 | PR with the QA evidence attached | 🤝 BOTH | `gh pr create --body-file` |
| 13 | Second opinion, approve, merge | 🧪 QA | `gh pr review --approve` — by a person |
| 14 | The same gates in CI | 🤝 BOTH | `.github/workflows/qa.yml` |

### 🧰 Before you start

| Need | Why | If missing |
|---|---|---|
| `git`, `python3`, `pandas` | the repo, the app, the data frame | `pip install pandas` |
| `sqlite3` CLI | the warehouse — stands in for Iceberg behind Trino | ships with macOS / Ubuntu; `brew install sqlite` |
| `pytest` | steps 5, 6, 9, 10 | `pip install pytest` |
| `claude` CLI | the seven `claude -p` calls | **required** — nothing here is canned |
| `gh` CLI | push and open the PR | **optional** — step 12–13 commands are printed, not run |
| the repo at `/Users/satyak/Ranagr_DataCatalog_Iceberg` | the system under test | clone it there first |

**Nothing in this notebook is faked.** Every gate, every RED run and every GREEN run was executed
against a scratch copy of the real repository before it was written down.

**Cell: 2**

### 🗺️ The whole QA pipeline, as a picture

![The QA pipeline: five lanes, top to bottom](qa_pipeline_overview.png)

Read it **down** for the hat changes and **across** inside a lane for the steps. The seat that
performs each step is the lane it sits in, and every `claude -p` box names the tool grant it runs
with — that grant is the difference between a reviewer, a test author and a developer.

<details>
<summary>The mermaid source for this diagram (edit here, then re-render)</summary>

```mermaid
flowchart TB
    subgraph SETUP["🧪 QA · the clean room"]
        direction LR
        B0["🧹 0 · copy the repo<br/>build the warehouse<br/>git init · baseline"] --> B1["▶️ 0 · run the legacy suite<br/>📎 baseline = GREEN"]
    end
    subgraph DEV["👨‍💻 Developer"]
        direction LR
        T["🎟️ 1 · ticket + branch"] --> F["🔨 2 · feature<br/>migration · planner · catalog"]
        F --> C["📝 3 · commit message<br/>claude -p · no tools"]
    end
    subgraph QA["🧪 QA Lead"]
        direction LR
        P["🗺️ 4 · test plan<br/>claude -p · Read,Grep,Glob"] --> W["✍️ 5 · test cases<br/>claude -p · Read,Write"]
        W --> R["🔴 6 · first run<br/>pytest → RED"]
        R --> D["🐞 7 · defect report<br/>claude -p · severity"]
    end
    subgraph FIX["👨‍💻 DEV fixes → 🧪 QA verifies"]
        direction LR
        X["🔧 8 · fix the root cause<br/>claude -p · Read,Edit"] --> G["🟢 9 · re-run<br/>GREEN + regression"]
        G --> M["🧬 10 · mutation<br/>grade the tests"]
    end
    subgraph SHIP["🤝 Sign-off"]
        direction LR
        S["🛡️ 11 · security review<br/>claude -p · blockers = 0"] --> PR["📬 12 · PR + evidence<br/>claude -p"]
        PR --> A["🤝 13 · approve + merge<br/>a person signs"]
        A --> CI["⚙️ 14 · CI runs<br/>the same gates"]
    end
    SETUP --> DEV
    DEV --> QA
    QA --> FIX
    FIX --> SHIP
```

Re-render after editing:

```bash
npx -y @mermaid-js/mermaid-cli@11.16.0 -i qa_pipeline_overview.mmd -o qa_pipeline_overview.png -b '#12172a' -s 3
```

</details>

**Cell: 3**

## 🧭 How to read the cells

**The markers you will see:**

| Marker | Means |
|---|---|
| 👨‍💻 `DEV` | the Developer performs this cell |
| 🧪 `QA` | the QA Lead performs this cell |
| 🎯 `THE CALL` | the one line that talks to Claude |
| 🚦 `THE GATE` | a check that stops the notebook if the evidence is wrong |
| 📎 `EVIDENCE` | output QA keeps and attaches to the PR |

**Run them top to bottom.** All state lives in files under `/tmp/qa_demo` — never in notebook
memory — so re-running a cell is safe. Step 0 rebuilds the scratch repo from the real one whenever
you want a clean slate. **Your real repository is never modified.**

**No `cd` anywhere.** Every path is absolute and git is told where to work:

```bash
git -C /tmp/qa_demo status      # -C means: run as if you were standing in this folder
```

**Every `claude -p` call ends `</dev/null`.** `%%bash` pipes the whole cell into bash on stdin and
`claude -p` reads stdin — without it the CLI swallows the rest of your cell and the commands below
it silently never run.

### 🚦 What makes a gate a gate

A gate is a command that **exits non-zero when the evidence is wrong**, placed as the **last line
of its own cell**. `%%bash` only raises on the last command, so a check in the middle of a cell is
a comment, not a gate.

Three rules QA never bends:

1. **Gate on the artifact, never on the exit code.** An agent that ran out of turns exits 1 with
   the work already on disk; an agent that decided to change nothing exits 0. Ask the file system.
2. **A gate that cannot fail is decoration.** Step 6 gates on the suite being **RED**. A suite that
   passes the moment you write it has not been shown to test anything.
3. **The fix must never touch the tests.** Step 8 gates on `git diff --quiet -- tests/`.

**Normalise before you gate.** Every step that asks for JSON says *output ONLY a JSON object* — and
across runs you will still get a ```json fence around it, or a sentence of commentary before the
opening brace. The fix is not a sterner prompt; it is one deterministic command that pins the shape
before the gate looks:

```python
p.write_text(s[s.index('{'):s.rindex('}') + 1])   # first brace to last brace
```

Prompts are persuasion and they hold most of the time. A pipeline gate has to hold **every** time,
so anything a gate reads gets normalised into a known shape first.

### 🎛️ The flags, and what QA buys with each

| Flag | What it buys you |
|---|---|
| `-p "…"` | headless: run once, print, exit — the only mode CI can use |
| `--model claude-sonnet-4-6` | pinned model → same verdict tomorrow, predictable cost |
| `--allowedTools ""` | grants nothing. The diff is in the prompt; it needs no disk |
| `--allowedTools "Read,Grep,Glob"` | **a reviewer**: may look, may not touch. QA's default |
| `--allowedTools "Read,Write"` | may create new files (the test file) — but not edit yours |
| `--allowedTools "Read,Edit"` + `--permission-mode acceptEdits` | **the developer seat**: may change existing files |
| `--add-dir /tmp/qa_demo` | lets the agent work in a folder outside the current one |
| `--max-turns N` | runaway protection — measured, not guessed (see below) |

### 🔢 Why the turn budgets look the way they do

`--max-turns` counts **moves** (read, edit, re-read, final message), not seconds and not tokens.
The budgets below were measured, not guessed: run the call once with a deliberately huge cap and
`--output-format json`, read `num_turns`, do that three times, take the **highest** and multiply by
about **2.5**.

| Step | Job | Cap | Why |
|---|---|---|---|
| 3, 7, 12 | write text, no tools | 8 | one move plus headroom |
| 4, 11 | read the repo and report | 12 | reading is cheap, wandering is not |
| 5 | write a new test file | 15 | measured max 10 for a write-a-file job |
| 8 | edit existing source | 15 | measured max 6, but edits invite re-reading |

A cap that is too low does not save money — the tokens were already spent when the plug was
pulled. It just throws away the ending and leaves half-finished work on disk.

### 🔁 Who holds the ball, when

The 15 steps are a relay between the two seats. **The builder never grades their own work** — every
DEV step hands its output to QA for judgement, and the one DEV step in the middle (the fix) is
sandwiched between QA finding the defect and QA verifying the fix:

```mermaid
flowchart LR
    A["🧪 QA<br/>0 · clean room + baseline"] --> B["👨‍💻 DEV<br/>1-3 · ticket → feature → commit"]
    B --> C["🧪 QA<br/>4-7 · plan → tests → RED → defect"]
    C --> D["👨‍💻 DEV<br/>8 · fix the root cause"]
    D --> E["🧪 QA<br/>9-11 · GREEN → mutation → review"]
    E --> F["🤝 BOTH<br/>12-14 · PR → merge → CI"]
```

**Why the developer never verifies their own fix:** the developer proves the feature *works* — and
the same eyes that missed the missing tag once will miss it again. Only a seat that did not build
the change goes looking for what *else* broke. That is not distrust of a person; it is distrust of
any single pair of eyes, including yours.


In [1]:
%%bash
# Cell: 4
# ============================================================================
# 🧪 QA - PREFLIGHT. Every tool this notebook needs, checked before anything runs.
# A missing tool discovered in step 8 costs more than a missing tool discovered now.
# ============================================================================

# The system under test must exist where we think it does.
ls -d /Users/satyak/Ranagr_DataCatalog_Iceberg

# The toolchain.
git --version
sqlite3 --version
python3 --version
python3 -c "import pandas; print('pandas', pandas.__version__)"
python3 -c "import pytest;  print('pytest', pytest.__version__)"

# The CLI that does the thinking. No fallback exists - if this line fails, stop here.
claude --version

/Users/satyak/Ranagr_DataCatalog_Iceberg
git version 2.39.2 (Apple Git-143)
3.39.5 2022-10-14 20:58:05 554764a6e721fab307c63a4f98cd958c8428a5d9d8edfde951858d6fd02daapl
Python 3.11.10
pandas 2.3.1
pytest 8.3.5
2.1.239 (Claude Code)


**Cell: 5**

---
## 🧹 Step 0 · The clean room — 🧪 **QA**

**At a glance**

- 👤 **Who does this:** 🧪 **QA** — the person doing the measuring owns the starting state, so no result can be blamed on "your machine".
- 🧪 **What this step does:** copies the repo to `/tmp/qa_demo`, builds the warehouse from `migrations/000`, commits a git baseline, runs the legacy suite.
- ✅ **What you should see:** the legacy suite **GREEN**, saved as `00_baseline.txt` — everything later is measured against it.
- ⚠️ **Common mistake:** skipping the baseline run. If you cannot show the suite was green *before* the change, you cannot claim the change turned it red.

**Why QA owns the environment, not the developer.** A test result is only worth something if you
can say what it was measured against. That means the starting state has to be identical every time
and owned by the person doing the measuring.

**Four things happen here, in this order, and the order is the point:**

1. **Copy** the real repository to `/tmp/qa_demo`. Your working copy is never touched — QA does not
   get to break the developer's machine, and the developer does not get to influence QA's run by
   leaving an edited file lying around.
2. **Build the warehouse.** The real repo ships its 16 customers as Python tuples — fine for a
   demo, wrong for a lesson about schema change. Step 0b materialises them into a database
   (`warehouse/polaris.db`, SQLite standing in for an Iceberg table behind Trino) from a versioned
   DDL file, `migrations/000_create_dimcustomer.sql`, and points `core/data.py` at it. The `.db`
   file is **not** committed; the migration that builds it is. This is the state the ticket is
   opened against: **a table that already exists, with data already in it.**
3. **`git init` + commit a baseline.** Now every later question — *what changed?* — has an exact
   answer that a command can print. Without this commit, "the diff" is an opinion.
4. **Run the existing suite and record that it passes.** This is the single most skipped step in
   real QA, and it is the one that saves the argument later: if you cannot show the suite was green
   *before* the change, you cannot claim the change turned it red.

That fourth artifact has a name — the **baseline**. Everything after this is measured against it.

**Cell: 6**

```mermaid
flowchart LR
    SRC["📁 real repo<br/>Ranagr_DataCatalog_Iceberg"] -->|copy, never move| TMP["🧪 /tmp/qa_demo"]
    TMP --> WH["🏗️ migrations/000 → sqlite3<br/>warehouse/polaris.db"]
    WH --> INIT["git init -b main<br/>commit baseline"]
    INIT --> BASE["▶️ run legacy suite"]
    BASE --> EV["📎 EVIDENCE<br/>baseline = GREEN"]
    EV --> OK{"green?"}
    OK -->|no| STOP["🛑 fix the repo first<br/>never test on a red baseline"]
    OK -->|yes| GO["✅ safe to change code"]
```

In [2]:
%%bash
# Cell: 7
# ============================================================================
# 🧪 QA - STEP 0: build the clean room
# ============================================================================

# Wipe any previous run. This cell is the "start over" button for the whole notebook.
rm -rf /tmp/qa_demo

# A copy, not a move. The real repository is read-only as far as this notebook is concerned.
# --exclude keeps compiled junk and any existing history out of the measurement.
mkdir -p /tmp/qa_demo
rsync -a --exclude '.git' --exclude '__pycache__' --exclude '.DS_Store' /Users/satyak/Ranagr_DataCatalog_Iceberg/ /tmp/qa_demo/

# Somewhere for QA to keep evidence. Everything attached to the PR later comes from this folder.
mkdir -p /tmp/qa_demo/qa_evidence

# Compiled bytecode must never be tracked, and this is not housekeeping - it is a gate that works.
# Later steps ask "did anything under core/ change?" to decide whether a fix landed. Running the
# app rewrites core/__pycache__/*.pyc, so if those files are tracked, git reports a change in core/
# every time anyone imports the module - and the gate answers yes when the answer is no.
# The warehouse file is a build product of migrations/*.sql - rebuilt, never committed.
printf '__pycache__/\n*.pyc\nwarehouse/*.db\n' > /tmp/qa_demo/.gitignore

ls /tmp/qa_demo

README.md
ambaricloud_logo.png
app.py
core
qa_evidence
requirements.txt
tests


**Cell: 8**

### 🏗️ Step 0b · The warehouse exists before the ticket — 🧪 **QA**

**At a glance**

- 🔹 **Who does this:** 🧪 **QA** — still building the clean room; the developer has not appeared yet.
- 📄 **What this step does:** materialises the 16 customers into `warehouse/polaris.db` from a versioned DDL file, so "add a column" means a **migration**, not a Python edit.
- 🔹 **What you should see:** `PRAGMA table_info` prints 10 columns; the catalog prints 10 columns **plus tags**.
- ⚠️ **Common mistake:** treating the two schemas as one. The migration updates the warehouse schema; **nothing updates the catalog** — that gap is the whole notebook.

**Where the data lives decides what "add a column" means.** In the repository as it ships, the
customer rows are a Python list and `load_table()` wraps them in a DataFrame. Add a column there
and the change is a code edit. In the platform this app stands in for, `dimcustomer` is an Iceberg
table behind Trino: adding a column is a **migration** — `ALTER TABLE … ADD COLUMN` — that someone
writes, versions, applies, and can diff. That is the shape QA-114 should have, so step 0 builds it.

**Two files, one command:**

| File | Role |
|---|---|
| `migrations/000_create_dimcustomer.sql` | the DDL and the 16 seed rows — **this is the table as it existed before anyone opened QA-114** |
| `core/data.py` | reads `SELECT *` from `warehouse/polaris.db`. It defines nothing; a new column arrives by migration, not by editing Python |
| `sqlite3 warehouse/polaris.db < migrations/000_*.sql` | replays the migration. Same command CI runs in step 14 |

**Two schemas from here on, and the difference is the whole notebook.** The warehouse has one —
`PRAGMA table_info(dimcustomer)` prints columns and types. The catalog in `core/catalog.py` has
another — columns, types and **classification tags**, and Ranger compiles its masks from that one.
A migration updates the first. Nothing updates the second.


In [3]:
%%bash
# Cell: 9
# ============================================================================
# 🧪 QA - STEP 0b: the warehouse. The table exists, with data, BEFORE the ticket.
# ============================================================================
mkdir -p /tmp/qa_demo/migrations /tmp/qa_demo/warehouse

# Migration 000: the table as it stood before QA-114. Versioned DDL, not a Python list.
cat > /tmp/qa_demo/migrations/000_create_dimcustomer.sql <<'EOF'
-- 000: polaris.market.dimcustomer, as it existed before anyone opened QA-114.
-- SQLite stands in for an Iceberg table behind Trino. One row per customer.
CREATE TABLE dimcustomer (
    customer_id INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT,
    state       TEXT,
    region      TEXT NOT NULL,      -- tenant key: US / EU (row-filter)
    email       TEXT,
    ssn         TEXT,
    card        TEXT,
    plan        TEXT,
    total_spend REAL
);

INSERT INTO dimcustomer
(customer_id, name, city, state, region, email, ssn, card, plan, total_spend) VALUES
(1001, 'Maria Lopez', 'Bolingbrook', 'IL', 'US', 'maria.lopez@example.com', '859-45-5012', '3657-2286-1679-8935', 'Gold', 14250.75),
(1002, 'James Carter', 'Bolingbrook', 'IL', 'US', 'james.carter@example.com', '421-88-7741', '4111-1111-1111-1234', 'Platinum', 22890.10),
(1003, 'Priya Nair', 'Bolingbrook', 'IL', 'US', 'priya.nair@example.com', '330-21-9087', '5500-0000-0000-0004', 'Gold', 19980.00),
(1004, 'Wei Zhang', 'Bolingbrook', 'IL', 'US', 'wei.zhang@example.com', '612-77-3344', '6011-0009-9013-9424', 'Silver', 8420.40),
(1005, 'Aisha Khan', 'Bolingbrook', 'IL', 'US', 'aisha.khan@example.com', '788-12-6655', '3782-8224-6310-005', 'Platinum', 31200.55),
(1006, 'David Cohen', 'Bolingbrook', 'IL', 'US', 'david.cohen@example.com', '205-66-1290', '4012-8888-8888-1881', 'Gold', 16730.25),
(1007, 'Sofia Rossi', 'Bolingbrook', 'IL', 'US', 'sofia.rossi@example.com', '551-09-4432', '4222-2222-2222-2225', 'Silver', 9510.00),
(1008, 'Liam Murphy', 'Bolingbrook', 'IL', 'US', 'liam.murphy@example.com', '147-33-8821', '5105-1051-0510-5100', 'Gold', 12740.90),
(1009, 'Hana Sato', 'Bolingbrook', 'IL', 'US', 'hana.sato@example.com', '962-40-1177', '3530-1113-3330-0000', 'Platinum', 27640.30),
(1010, 'Omar Haddad', 'Bolingbrook', 'IL', 'US', 'omar.haddad@example.com', '418-55-7203', '6304-0000-0000-0000', 'Gold', 15310.65),
(1011, 'Grace Lee', 'Bolingbrook', 'IL', 'US', 'grace.lee@example.com', '377-19-5560', '4917-4842-3535-0001', 'Silver', 7020.10),
(1012, 'Noah Becker', 'Naperville', 'IL', 'US', 'noah.becker@example.com', '880-23-4419', '4001-9192-5753-7193', 'Gold', 11890.00),
(1013, 'Emma Wilson', 'Aurora', 'IL', 'US', 'emma.wilson@example.com', '263-71-9902', '5200-8282-8282-8210', 'Platinum', 20410.85),
(1014, 'Lucas Meyer', 'Munich', 'BY', 'EU', 'lucas.meyer@example.de', '551-02-3380', '4000-0566-5566-5556', 'Gold', 17850.20),
(1015, 'Chloe Dubois', 'Paris', '75', 'EU', 'chloe.dubois@example.fr', '204-88-1142', '5555-5555-5555-4444', 'Silver', 9930.45),
(1016, 'Mateo Garcia', 'Madrid', '28', 'EU', 'mateo.garcia@example.es', '709-31-6650', '4242-4242-4242-4242', 'Platinum', 24560.00);
EOF

# core/data.py now READS the warehouse. It no longer owns the rows or the column list.
cat > /tmp/qa_demo/core/data.py <<'EOF'
"""Warehouse access — stands in for an Iceberg scan through Trino.

polaris.market.dimcustomer lives in a SQLite database at warehouse/polaris.db,
built by replaying migrations/*.sql in order (000 creates and seeds it). This
module reads; it never defines the schema. A new column arrives here through a
migration, not by editing Python.
"""
from __future__ import annotations
from contextlib import closing
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path(__file__).resolve().parent.parent / "warehouse" / "polaris.db"

_TABLES = {"polaris.market.dimcustomer": "dimcustomer"}


def load_table(fqn: str) -> pd.DataFrame:
    """Stand-in for an Iceberg scan on S3. Returns the full raw table."""
    if fqn not in _TABLES:
        raise KeyError(f"no warehouse table for {fqn}")
    if not DB_PATH.exists():
        raise FileNotFoundError(f"{DB_PATH} missing - replay migrations/*.sql to build it")
    with closing(sqlite3.connect(DB_PATH)) as con:
        return pd.read_sql_query(f"SELECT * FROM {_TABLES[fqn]}", con)
EOF

# Build it. This exact line is what CI runs in step 14, once per migration file, in order.
rm -f /tmp/qa_demo/warehouse/polaris.db
sqlite3 /tmp/qa_demo/warehouse/polaris.db < /tmp/qa_demo/migrations/000_create_dimcustomer.sql

# The table, as the warehouse sees it: 10 columns, 16 rows.
sqlite3 -header -column /tmp/qa_demo/warehouse/polaris.db "PRAGMA table_info(dimcustomer);"
sqlite3 /tmp/qa_demo/warehouse/polaris.db "SELECT count(*) AS customers FROM dimcustomer;"

# 🚦 THE GATE - the app reads the warehouse, not a Python list. 16 rows, 10 columns, or stop.
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core.data import load_table, DB_PATH
df = load_table('polaris.market.dimcustomer')
print('load_table ->', df.shape, 'from', DB_PATH.name)
assert df.shape == (16, 10), df.shape"


cid  name         type     notnull  dflt_value  pk
---  -----------  -------  -------  ----------  --
0    customer_id  INTEGER  0                    1 
1    name         TEXT     1                    0 
2    city         TEXT     0                    0 
3    state        TEXT     0                    0 
4    region       TEXT     1                    0 
5    email        TEXT     0                    0 
6    ssn          TEXT     0                    0 
7    card         TEXT     0                    0 
8    plan         TEXT     0                    0 
9    total_spend  REAL     0                    0 
16
load_table -> (16, 10) from polaris.db


In [4]:
%%bash
# Cell: 10
# 🧪 QA - STEP 0c: history starts here, so "what changed" is answerable by a command.
# The migration is tracked. The .db it builds is not (see .gitignore) - like any build product.
git -C /tmp/qa_demo init -b main -q
git -C /tmp/qa_demo config user.email "qa@ambaricloud.com"
git -C /tmp/qa_demo config user.name  "QA Lead"
git -C /tmp/qa_demo add -A
git -C /tmp/qa_demo commit -qm "chore: baseline of the app under test, warehouse built from migrations/000"

# What we are testing.
git -C /tmp/qa_demo log --oneline
git -C /tmp/qa_demo ls-files migrations core
git -C /tmp/qa_demo status --short --ignored | grep warehouse


bc532e5 chore: baseline of the app under test, warehouse built from migrations/000
core/__init__.py
core/catalog.py
core/data.py
core/guardrails.py
core/identity.py
core/mcp_server.py
core/planner.py
core/ranger.py
migrations/000_create_dimcustomer.sql
!! warehouse/


In [5]:
%%bash
# Cell: 11
cat /tmp/qa_demo/tests/test_flow.py

"""Headless smoke test — proves the full flow without Streamlit.

Run:  python tests/test_flow.py     (exit 0 = pass)

Asserts the slide-22 behavior:
  • the misspelled 'Bollingbrook' resolves to Bolingbrook
  • admin sees raw, all tenants
  • analyst sees SHA-256-hashed PII, card redacted, US rows only
  • support sees last-4 SSN, card redacted, EU rows only
  • no raw SSN/card leaks for non-admins (guardrails)
"""
import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from core import login, mcp_server
from core.planner import plan_sql

Q = "Top 10 Customers who live in Bollingbrook"
SSN_RAW = "859-45-5012"


def run(user):
    p = login(user)
    plan = plan_sql(Q)
    df, decision, audit = mcp_server.run_query(p, plan)
    return p, plan, df, decision, audit


def main():
    # planner correctness
    plan = plan_sql(Q)
    assert plan.filters.get("city") == "Bolingbrook", plan.filters
    assert plan.limit == 10 and plan.order_by == "total_

In [6]:
%%bash
# Cell: 12
# 📎 EVIDENCE + 🚦 THE GATE - the baseline must be GREEN before anyone changes a line.
# The app ships a headless smoke test that proves the three-identity governance story.
# We keep its output; the PR will reference it as "baseline, before the change".
python3 /tmp/qa_demo/tests/test_flow.py > /tmp/qa_demo/qa_evidence/00_baseline.txt 2>&1
cat /tmp/qa_demo/qa_evidence/00_baseline.txt

# Last line of its own cell. Non-zero here means the repo was already broken, and nothing
# measured after this point would mean anything.
python3 /tmp/qa_demo/tests/test_flow.py > /dev/null 2>&1

PLAN  : SELECT customer_id, name, city, region, email, ssn, card, total_spend FROM polaris.market.dimcustomer WHERE city = 'Bolingbrook' ORDER BY total_spend DESC LIMIT 10;
ADMIN : rows=10 raw_ssn=yes masked=[]
ANALYST: rows=10 ssn[0]=2B92EDFE70B9C924… filter=region = 'US'
SUPPORT: rows=0 ssn[sample]=XXXXX... filter=region = 'EU'

ALL CHECKS PASSED ✅


**Cell: 13**

### 👀 Optional · Look at the thing you are about to test

**At a glance**

- 🧪 **Who does this:** 🧪 **QA**, optionally — seeing the app makes the tests mean something.
- 🔹 **What you should see:** now — `email`/`ssn`/`card` masked per role; after step 2 — **raw phone numbers** for alice and carol; after step 8 — masked again.
- ⚠️ **Common mistake:** starting it against the original repo. Point it at `/tmp/qa_demo` or you will never see the defect appear.

Everything in this notebook is a command, and commands are easy to trust too little. The system
under test is a **Streamlit app** — start it and click through the same flow the tests exercise:
pick an identity, ask the question, read the SQL, approve it, and watch Ranger mask the answer.

**Start it against `/tmp/qa_demo`, not the original repo.** That single choice turns the app into a
live instrument for this lesson:

| When you open it | What you will see |
|---|---|
| now, before step 2 | the app as it ships — `email`, `ssn`, `card` masked per role |
| after step 2 | a **`phone` column with raw numbers** for alice and carol. The defect, in a browser |
| after step 8 | the same column, hashed for alice and `XXXXX0114` for carol |

Switching the identity in the sidebar is the fastest explanation of row-filtering and column-masking
anyone has ever given you: `dave` sees 16 rows raw, `alice` sees 13 US rows hashed, `carol` sees
3 EU rows with last-4.

**It runs in the background,** so the cell returns and the rest of the notebook keeps working. The
last line asks the server whether it is actually serving — a port that is open but dead is worse
than one that never opened.

In [ ]:
%%bash
# Cell: 14
# ============================================================================
# 👀 OPTIONAL - start the app under test (Streamlit), in the background
# ============================================================================
# Port 8521 - the rest of the Mastering series lives on 8510-8520.
# Killing any previous copy first makes this cell safe to re-run, which is the same rule
# every other cell in this notebook follows.
pkill -f "streamlit run /tmp/qa_demo/app.py" 2>/dev/null

# nohup + & so the server outlives the cell. The log is a file, like every other artifact here.
nohup streamlit run /tmp/qa_demo/app.py \
  --server.port 8521 \
  --server.address 0.0.0.0 \
  --server.headless true \
  --server.enableCORS false \
  --server.enableXsrfProtection false \
  --server.enableWebsocketCompression false \
  --browser.serverAddress 10.0.0.154 > /tmp/qa_demo/qa_evidence/app.log 2>&1 &

# Give it a moment to bind the port before asking whether it works.
sleep 6
echo "open  ->  http://10.0.0.154:8521"

# 🚦 THE GATE - a port that is open but not serving is worse than one that never opened.
# --fail makes curl exit non-zero on an HTTP error, so this line is a real check.
curl -sS --fail -o /dev/null http://10.0.0.154:8521
echo "the app is serving"

In [ ]:
%%bash
# Cell: 15
# 👀 Stop it when you are done. Streamlit holds the port until you do.
pkill -f "streamlit run /tmp/qa_demo/app.py"
echo "app stopped"

**Cell: 16**

---
## 🎟️ Step 1 · The ticket and the branch — 👨‍💻 **DEV**

Hat change. You are the developer now.

**At a glance**

- 🌿 **Who does this:** 👨‍💻 **DEV** — the ticket, the branch and the feature belong to the builder's seat.
- 🗺️ **What this step does:** writes ticket QA-114 down as a file in the repo, then cuts `feature/qa-114-phone`.
- 🌿 **What you should see:** `tickets/QA-114.md` in the repo, and git on the feature branch — `main` stays shippable.
- ⚠️ **Common mistake:** vague acceptance criteria. "Add phone support" produces vague tests; the ticket below names observable behaviour **per role**.

**Write the ticket down as a file in the repo.** Not because process demands it, but because in
step 4 QA hands that exact file to Claude as the statement of intent. A test plan built from a
diff alone can only ask *"is this code correct?"* — the interesting question is
*"does this code do what was asked, and what else did it touch?"*, and only the ticket can answer
the first half.

**Acceptance criteria are QA's contract.** Vague ones ("add phone support") produce vague tests.
The ticket below names the observable behaviour for **each of the three roles**, because in this
application role is the axis that everything turns on.

**Cell: 17**

```mermaid
flowchart LR
    TICKET["🎟️ QA-114.md<br/>intent + acceptance criteria"] --> BR["🌿 git switch -c<br/>feature/qa-114-phone"]
    BR --> WHY["main stays shippable<br/>the branch carries the risk"]
    TICKET -.->|read again in step 4| PLAN["🧪 QA test plan"]
    TICKET -.->|read again in step 12| PRB["📬 PR body"]
```

In [7]:
%%bash
# Cell: 18
# ============================================================================
# 👨‍💻 DEV - STEP 1: the ticket, then the branch
# ============================================================================

# The ticket is a file in the repo. QA reads it in step 4; the PR body quotes it in step 12.
# HEREDOC: everything between <<'EOF' and EOF is written exactly as typed. Nothing runs.
mkdir -p /tmp/qa_demo/tickets
cat > /tmp/qa_demo/tickets/QA-114.md <<'EOF'
# QA-114 - Add customer phone number to the profile

## Why
Support agents handling an escalation have no way to call the customer back. Today they read the
customer profile, find an email address, and wait a day for a reply.

## What
Add a `phone` column to `polaris.market.dimcustomer` and include it in the default customer
projection, so it appears alongside name / city / email in the app's results.

## Acceptance criteria
1. `phone` is present in the seed data for every one of the 16 customers.
2. `phone` appears in the default selected columns of the planner.
3. `dave` (admin) sees the raw phone number - admins are entitled to raw values.
4. `alice` (analyst) must NOT see a raw phone number.
5. `carol` (support) must NOT see a raw phone number, but must see enough of it to confirm
   identity on a call.
6. Nothing about the existing email / ssn / card behaviour changes.

## Out of scope
Dialling, SMS, phone number validation, international formatting rules.
EOF

# Branch first, then change code. main stays shippable at all times.
git -C /tmp/qa_demo switch -c feature/qa-114-phone

cat /tmp/qa_demo/tickets/QA-114.md

Switched to a new branch 'feature/qa-114-phone'


# QA-114 - Add customer phone number to the profile

## Why
Support agents handling an escalation have no way to call the customer back. Today they read the
customer profile, find an email address, and wait a day for a reply.

## What
Add a `phone` column to `polaris.market.dimcustomer` and include it in the default customer
projection, so it appears alongside name / city / email in the app's results.

## Acceptance criteria
1. `phone` is present in the seed data for every one of the 16 customers.
2. `phone` appears in the default selected columns of the planner.
3. `dave` (admin) sees the raw phone number - admins are entitled to raw values.
4. `alice` (analyst) must NOT see a raw phone number.
5. `carol` (support) must NOT see a raw phone number, but must see enough of it to confirm
   identity on a call.
6. Nothing about the existing email / ssn / card behaviour changes.

## Out of scope
Dialling, SMS, phone number validation, international formatting rules.


**Cell: 19**

---
## 🔨 Step 2 · Build the feature — 👨‍💻 **DEV**

**At a glance**

- 🔹 **Who does this:** 👨‍💻 **DEV** — building the feature, by hand, the way a hurried engineer would.
- 🔹 **What this step does:** one `ALTER TABLE` migration plus two one-line edits (planner, catalog).
- ⚠️ **What you should see:** the app runs and `phone` shows up — the feature **works**. Nothing looks wrong yet.
- ⚠️ **Common mistake / the planted defect:** the catalog line carries **no `tag="PII"`**. Syntactically perfect, semantically wrong — and no tool in the repo will say so until step 6.

Three files. Read them as a developer would, quickly, at the end of a Thursday.

| File | The change | Feels like |
|---|---|---|
| `migrations/001_qa114_add_phone.sql` | `ALTER TABLE dimcustomer ADD COLUMN phone` + a backfill | a migration, like the one before it |
| `core/planner.py` | `"phone"` added to `_DEFAULT_COLS` | one word |
| `core/catalog.py` | `Column("phone", "string")` declared | one line, obviously right |

**`core/data.py` does not change.** It reads `SELECT *`; the new column arrives on its own the
moment the migration is applied. That is the point of a warehouse — and the reason the developer
will look at the data, see `phone`, and feel done.

**This is written by hand, not by `claude -p`, and that is deliberate:**

- ▶️ The bug has to be the **same bug every time** you run the notebook, or the QA steps have nothing to find.
- 🔹 It is also the most honest version of the story: this defect does not come from an AI writing bad code. It comes from a competent engineer adding a column and declaring it exactly the way the nine columns above it are declared.

**Look hard at the catalog line before you run the cell:**

- 📄 Every other classified column in that file carries a tag — `tag="PII"`, `tag="PCI"`, `tag="TENANT"`.
- 🔹 Whether a phone number is personal data is a *judgement*, and judgement is precisely what a hurried developer defers.
- 🧪 Nothing in Python, nothing in the type system, and nothing in the test suite as it stands today will ask.

That is the shape of the most expensive class of defect in a governed data platform: **not wrong
code — absent metadata.** The code does exactly what it says. Nobody said the important part.

**Cell: 20**

```mermaid
flowchart TD
    D["migrations/001<br/>ALTER TABLE ADD COLUMN phone"] --> WH["warehouse/polaris.db<br/>11 columns"] --> APP
    P["core/planner.py<br/>+ phone in _DEFAULT_COLS"] --> APP
    C["core/catalog.py<br/>Column phone string"] --> APP["the app runs<br/>✅ feature works"]
    C -.->|no tag=PII| GAP["🕳️ classification gap"]
    GAP --> R["ranger.enforce masks only<br/>classified_columns"]
    R --> LEAK["😱 analyst + support<br/>see raw phone numbers"]
    APP -.->|dev sees this| OK["looks finished"]
    LEAK -.->|only a test sees this| QA["🧪 step 6"]
```

**Cell: 21**

### 📋 Step 2 · Schema **before** the change — 🧪 **QA** · two schemas

- 🔹 **Who does this:** 🧪 **QA** — capturing the "before" picture so the "after" can be diffed, not remembered.
- 🔹 **What you should see:** 10 columns in both views; only the catalog has a **tag** column — watch it.

Two views of the same table, both saved as evidence so step 2d can diff them:

| View | Command | What it knows |
|---|---|---|
| the warehouse | `PRAGMA table_info(dimcustomer)` | columns and types — what `ALTER TABLE` changes |
| the catalog | `catalog.describe()` | columns, types and **tags** — what Ranger compiles masks from |

10 columns in each. The **tag** column exists in only one of them, and it is the one to watch.


In [8]:
%%bash
# Cell: 22
# ============================================================================
# 📋 SCHEMA - BEFORE the change. Two schemas: the warehouse's and the catalog's.
# ============================================================================
# 1. The warehouse. What ALTER TABLE will change.
echo "--- warehouse: PRAGMA table_info(dimcustomer) ---"
sqlite3 -header -column /tmp/qa_demo/warehouse/polaris.db "PRAGMA table_info(dimcustomer);" \
  | tee /tmp/qa_demo/qa_evidence/02_db_schema_before.txt

# 2. The catalog. What describe_table() hands the agent, and what Ranger policies are compiled
#    from. Keep the tag column in view: it is the only thing that will matter in step 6.
echo; echo "--- catalog: describe_table() ---"
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core import catalog
d = catalog.describe('polaris.market.dimcustomer')
print(d['table'], '-', d['grain'])
print(f\"{'#':>2}  {'column':<12} {'type':<8} {'tag':<8} classified\")
for i, c in enumerate(d['columns'], 1):
    print(f\"{i:>2}  {c['name']:<12} {c['type']:<8} {c['tag'] or '-':<8} {'yes' if c['classified'] else 'no'}\")
print('classified_columns:', d['classified_columns'])
" | tee /tmp/qa_demo/qa_evidence/02_schema_before.txt


--- warehouse: PRAGMA table_info(dimcustomer) ---
cid  name         type     notnull  dflt_value  pk
---  -----------  -------  -------  ----------  --
0    customer_id  INTEGER  0                    1 
1    name         TEXT     1                    0 
2    city         TEXT     0                    0 
3    state        TEXT     0                    0 
4    region       TEXT     1                    0 
5    email        TEXT     0                    0 
6    ssn          TEXT     0                    0 
7    card         TEXT     0                    0 
8    plan         TEXT     0                    0 
9    total_spend  REAL     0                    0 

--- catalog: describe_table() ---
polaris.market.dimcustomer - one row per customer
 #  column       type     tag      classified
 1  customer_id  bigint   -        no
 2  name         string   -        no
 3  city         string   -        no
 4  state        string   -        no
 5  region       string   TENANT   no
 6  email        

In [9]:
%%bash
# Cell: 23
# ============================================================================
# 👨‍💻 DEV - STEP 2a: the migration. This is the ALTER statement.
# ============================================================================
# Forward-only, numbered after the one that created the table. ADD COLUMN, then backfill:
# US customers get a 630/331 number, EU customers an international one, so the row-filter
# story stays visible in the data itself.
cat > /tmp/qa_demo/migrations/001_qa114_add_phone.sql <<'EOF'
-- 001 / QA-114: support needs to call customers back.
-- Adds one column to polaris.market.dimcustomer and backfills the 16 existing rows.
ALTER TABLE dimcustomer ADD COLUMN phone TEXT;

UPDATE dimcustomer SET phone = CASE customer_id
    WHEN 1001 THEN '630-555-0101'
    WHEN 1002 THEN '630-555-0102'
    WHEN 1003 THEN '630-555-0103'
    WHEN 1004 THEN '630-555-0104'
    WHEN 1005 THEN '630-555-0105'
    WHEN 1006 THEN '630-555-0106'
    WHEN 1007 THEN '630-555-0107'
    WHEN 1008 THEN '630-555-0108'
    WHEN 1009 THEN '630-555-0109'
    WHEN 1010 THEN '630-555-0110'
    WHEN 1011 THEN '630-555-0111'
    WHEN 1012 THEN '331-555-0112'
    WHEN 1013 THEN '331-555-0113'
    WHEN 1014 THEN '+49-89-555-0114'
    WHEN 1015 THEN '+33-1-555-0115'
    WHEN 1016 THEN '+34-91-555-0116'
END;
EOF

# Apply it. Same command step 0b used for 000; same command CI will use in step 14.
sqlite3 /tmp/qa_demo/warehouse/polaris.db < /tmp/qa_demo/migrations/001_qa114_add_phone.sql

# The warehouse now has the column, and every row has a value.
sqlite3 -header -column /tmp/qa_demo/warehouse/polaris.db \
  "SELECT customer_id, name, phone, region FROM dimcustomer LIMIT 3;"
sqlite3 /tmp/qa_demo/warehouse/polaris.db \
  "SELECT count(*) AS rows_without_phone FROM dimcustomer WHERE phone IS NULL;"

# 🚦 A shape check the developer SHOULD do and usually does: 16 rows, 11 columns, no gaps -
# through load_table(), the door the app uses. core/data.py was not edited; SELECT * just grew.
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core.data import load_table
df = load_table('polaris.market.dimcustomer')
print('shape', df.shape, 'nulls', int(df.isna().sum().sum()))
assert df.shape == (16, 11) and df.isna().sum().sum() == 0
print(df[['customer_id','name','phone','region']].head(3).to_string(index=False))"


customer_id  name          phone         region
-----------  ------------  ------------  ------
1001         Maria Lopez   630-555-0101  US    
1002         James Carter  630-555-0102  US    
1003         Priya Nair    630-555-0103  US    
0
shape (16, 11) nulls 0
 customer_id         name        phone region
        1001  Maria Lopez 630-555-0101     US
        1002 James Carter 630-555-0102     US
        1003   Priya Nair 630-555-0103     US


In [10]:
%%bash
# Cell: 24
# 👨‍💻 DEV - STEP 2b: the planner selects it, so the column reaches the user.
sed -i '' 's|"card", "total_spend"\]|"card", "phone", "total_spend"]|' /tmp/qa_demo/core/planner.py
grep -n '_DEFAULT_COLS' /tmp/qa_demo/core/planner.py

40:_DEFAULT_COLS = ["customer_id", "name", "city", "region", "email", "ssn", "card", "phone", "total_spend"]
52:    plan = PlanResult(table=table, columns=[c for c in _DEFAULT_COLS if c in valid], notes=notes)


In [11]:
%%bash
# Cell: 25
# 👨‍💻 DEV - STEP 2c: declare it in the catalog, exactly like the line above it.
#
# Read this line next to its neighbours before you run the cell:
#     Column("email",       "string", tag="PII"),
#     Column("ssn",         "string", tag="PII"),
#     Column("card",        "string", tag="PCI"),
#     Column("plan",        "string"),
#     Column("phone",       "string"),          <- the line we are adding
#
# It is syntactically perfect and semantically wrong, and no tool in this repo will say so.
sed -i '' 's|        Column("plan",        "string"),|        Column("plan",        "string"),\n        Column("phone",       "string"),|' /tmp/qa_demo/core/catalog.py
sed -n '40,54p' /tmp/qa_demo/core/catalog.py

    columns=[
        Column("customer_id", "bigint", note="surrogate key"),
        Column("name",        "string"),
        Column("city",        "string"),
        Column("state",       "string"),
        Column("region",      "string", tag="TENANT", note="row-filter key (US/EU)"),
        Column("email",       "string", tag="PII"),
        Column("ssn",         "string", tag="PII"),
        Column("card",        "string", tag="PCI"),
        Column("plan",        "string"),
        Column("phone",       "string"),
        Column("total_spend", "double", note="lifetime spend"),
    ],
)



**Cell: 26**

### 📋 Step 2d · Schema **after** the change — 🧪 **QA** · diff both

- 📁 **Who does this:** 🧪 **QA** — same two commands as before, diffed against the saved evidence.
- 🌿 **What you should see:** warehouse diff `10  phone  TEXT`; catalog diff `phone  string  -  no` — a column with **no tag**, and nothing failing *yet*.

Same two commands. The warehouse diff is one line: `10  phone  TEXT`. The catalog diff is one line:
`phone  string  -  no`. The first is what the migration did. The second is what nobody did —
`classified_columns` did not grow. The app will run and the developer's check below will pass.
Nothing here is wrong *yet*.


In [12]:
%%bash
# Cell: 27
# ============================================================================
# 📋 SCHEMA - AFTER the change. 11 columns in both. Count the tags again.
# ============================================================================
echo "--- warehouse: PRAGMA table_info(dimcustomer) ---"
sqlite3 -header -column /tmp/qa_demo/warehouse/polaris.db "PRAGMA table_info(dimcustomer);" \
  | tee /tmp/qa_demo/qa_evidence/02_db_schema_after.txt

echo; echo "--- catalog: describe_table() ---"
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core import catalog
d = catalog.describe('polaris.market.dimcustomer')
print(d['table'], '-', d['grain'])
print(f\"{'#':>2}  {'column':<12} {'type':<8} {'tag':<8} classified\")
for i, c in enumerate(d['columns'], 1):
    print(f\"{i:>2}  {c['name']:<12} {c['type']:<8} {c['tag'] or '-':<8} {'yes' if c['classified'] else 'no'}\")
print('classified_columns:', d['classified_columns'])
" | tee /tmp/qa_demo/qa_evidence/02_schema_after.txt

echo; echo "--- warehouse, before -> after: what ALTER TABLE did ---"
diff /tmp/qa_demo/qa_evidence/02_db_schema_before.txt /tmp/qa_demo/qa_evidence/02_db_schema_after.txt || true

echo; echo "--- catalog, before -> after: what the developer did ---"
diff /tmp/qa_demo/qa_evidence/02_schema_before.txt /tmp/qa_demo/qa_evidence/02_schema_after.txt || true
# One line added in each. In the catalog: tag '-', classified 'no'. classified_columns did not grow.
# The app will run. The developer's check below will pass. Nothing here is wrong - yet.


--- warehouse: PRAGMA table_info(dimcustomer) ---
cid  name         type     notnull  dflt_value  pk
---  -----------  -------  -------  ----------  --
0    customer_id  INTEGER  0                    1 
1    name         TEXT     1                    0 
2    city         TEXT     0                    0 
3    state        TEXT     0                    0 
4    region       TEXT     1                    0 
5    email        TEXT     0                    0 
6    ssn          TEXT     0                    0 
7    card         TEXT     0                    0 
8    plan         TEXT     0                    0 
9    total_spend  REAL     0                    0 
10   phone        TEXT     0                    0 

--- catalog: describe_table() ---
polaris.market.dimcustomer - one row per customer
 #  column       type     tag      classified
 1  customer_id  bigint   -        no
 2  name         string   -        no
 3  city         string   -        no
 4  state        string   -        no
 5  

In [13]:
%%bash
# Cell: 28
# 👨‍💻 DEV - the developer's own check: does the feature work? Run the app's flow.
# It does. The phone number is right there in the output, exactly as the ticket asked.
# This is the moment a change gets pushed in real life.
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core import login, mcp_server
from core.planner import plan_sql
p = login('dave')
df, dec, aud = mcp_server.run_query(p, plan_sql('Top 10 Customers who live in Bollingbrook'))
print(df[['name','city','phone','total_spend']].head(5).to_string(index=False))
print('\n✅ QA-114 acceptance criteria 1, 2 and 3: phone is present, selected, raw for admin.')"

        name        city        phone  total_spend
  Aisha Khan Bolingbrook 630-555-0105     31200.55
   Hana Sato Bolingbrook 630-555-0109     27640.30
James Carter Bolingbrook 630-555-0102     22890.10
  Priya Nair Bolingbrook 630-555-0103     19980.00
 David Cohen Bolingbrook 630-555-0106     16730.25

✅ QA-114 acceptance criteria 1, 2 and 3: phone is present, selected, raw for admin.


In [14]:
%%bash
# Cell: 29
# 🚦 THE GATE - three files changed, and the notebook can say which.
# The migration is a NEW file: `git diff` ignores untracked files, so stage it first, then diff
# the feature paths against HEAD. (The ticket and qa_evidence/ are new too - they are not the
# feature, so they are kept out of this count.)
# An empty diff here means a sed did not match and every step below would be theatre.
git -C /tmp/qa_demo add -- migrations/
git -C /tmp/qa_demo diff HEAD --stat -- core/ migrations/

# Last line of its own cell: a diff that does not show exactly three files stops the run.
# Note what is NOT in the list: core/data.py. The warehouse changed; the reader of it did not.
git -C /tmp/qa_demo diff HEAD --stat -- core/ migrations/ | grep -q "3 files changed"

 core/catalog.py                    |  1 +
 core/planner.py                    |  2 +-
 migrations/001_qa114_add_phone.sql | 22 ++++++++++++++++++++++
 3 files changed, 24 insertions(+), 1 deletion(-)


**Cell: 30**

---
## 📝 Step 3 · The commit message — 👨‍💻 **DEV** · `claude -p` call 1 of 7

**At a glance**

- 🔹 **Who does this:** 👨‍💻 **DEV** — the builder describes their own change; QA will judge it later.
- 🌿 **What this step does:** stages the change, hands the staged diff to `claude -p`, gates the reply, commits.
- 🌿 **What you should see:** a Conventional Commit subject (`feat: …`) in `COMMIT_MSG.txt`, then a clean commit.
- ⚠️ **Common mistake:** diffing the working tree. `git diff` shows what you have *not* committed; only `git diff --cached` describes this commit.

**The `claude -p` recipe — call 1 of 7**

- 🌿 **The call:** `prompts/commit.md` (instructions) + `/tmp/staged.diff` (data), glued into one prompt.
- 🛡️ **The grant:** `--allowedTools ""` — grants nothing. The diff is already in the prompt, and an agent that cannot read your disk cannot leak it.
- 🚦 **The gate:** `grep "^feat\|^fix\|^chore" COMMIT_MSG.txt` — prints the subject only if the reply starts like a Conventional Commit; anything else exits non-zero and stops the cell before `git commit -F` runs.

**The shape**

```bash
claude -p "$(cat prompts/commit.md)$(cat /tmp/staged.diff)" --allowedTools "" </dev/null
```

**Why QA cares about a commit message.** Two reasons, both practical: the message is the first
thing the reviewer in step 11 reads, and — more importantly — a message that describes the change
honestly is the thing that lets someone bisect a regression six months from now. A history of
`fix stuff` commits turns every future investigation into a code read.

**Stage first, then diff.** `git diff` shows what you have *not* committed. `git diff --cached`
shows what you *are* committing. Only the second one describes this commit.

**`--allowedTools ""` grants nothing.** The diff is already inside the prompt, so the agent has no
reason to touch your disk — and an agent that cannot read your disk cannot leak it.

**The gate is the interesting cell:**

```bash
grep "^feat\|^fix\|^chore" COMMIT_MSG.txt
```

It prints the subject **only if** the reply starts like a Conventional Commit. An API error, an
apology, a stray code fence — none of those match, `grep` exits non-zero, and because it is the
last line of the cell the notebook stops before `git commit -F` can write nonsense into history.

**Cell: 31**

```mermaid
flowchart LR
    ADD["git add -A"] --> DIFF["git diff --cached<br/>> /tmp/staged.diff"]
    DIFF --> PR["📄 prompts/commit.md<br/>the instruction half"]
    PR --> CALL["🎯 claude -p<br/>--allowedTools ''"]
    CALL --> MSG["COMMIT_MSG.txt"]
    MSG --> GATE{"🚦 starts feat/fix/chore?"}
    GATE -->|no| STOP["🛑 stop<br/>history stays clean"]
    GATE -->|yes| COMMIT["git commit -F"]
```

In [15]:
%%bash
# Cell: 32
# ============================================================================
# 👨‍💻 DEV - STEP 3: claude -p writes the commit message
# ============================================================================

# Stage the change: the message must describe what is STAGED, not what is lying around.
git -C /tmp/qa_demo add -A

# The data half of the prompt.
git -C /tmp/qa_demo diff --cached > /tmp/staged.diff
wc -l /tmp/staged.diff

# The instruction half, in a file you can review and diff like any other source.
mkdir -p /tmp/qa_demo/prompts
cat > /tmp/qa_demo/prompts/commit.md <<'EOF'
You write git commit messages. Conventional Commits format. Subject line max 72 characters,
imperative mood, no trailing period. Then a blank line, then at most 3 bullets explaining WHY,
not what. Reference the ticket id QA-114 in the body. Output the message only - no preamble,
no code fences.

Write the commit message for this staged diff:

EOF


     168 /tmp/staged.diff


In [16]:
%%bash
# Cell: 33
cat /tmp/qa_demo/prompts/commit.md

You write git commit messages. Conventional Commits format. Subject line max 72 characters,
imperative mood, no trailing period. Then a blank line, then at most 3 bullets explaining WHY,
not what. Reference the ticket id QA-114 in the body. Output the message only - no preamble,
no code fences.

Write the commit message for this staged diff:



In [17]:
%%bash
# Cell: 34
cat /tmp/staged.diff

diff --git a/core/catalog.py b/core/catalog.py
index b39dd38..460050e 100644
--- a/core/catalog.py
+++ b/core/catalog.py
@@ -47,6 +47,7 @@ DIMCUSTOMER = Table(
         Column("ssn",         "string", tag="PII"),
         Column("card",        "string", tag="PCI"),
         Column("plan",        "string"),
+        Column("phone",       "string"),
         Column("total_spend", "double", note="lifetime spend"),
     ],
 )
diff --git a/core/planner.py b/core/planner.py
index 81590cd..10dd8c0 100644
--- a/core/planner.py
+++ b/core/planner.py
@@ -37,7 +37,7 @@ class PlanResult:
         return sql
 
 
-_DEFAULT_COLS = ["customer_id", "name", "city", "region", "email", "ssn", "card", "total_spend"]
+_DEFAULT_COLS = ["customer_id", "name", "city", "region", "email", "ssn", "card", "phone", "total_spend"]
 
 
 def plan_sql(question: str) -> PlanResult:
diff --git a/migrations/001_qa114_add_phone.sql b/migrations/001_qa114_add_phone.sql
new file mode 100644
index 0000000..d73f919
--- /dev/nu

In [18]:
%%bash
# Cell: 35
# 🎯 THE CALL. Two files glued into one prompt; the reply goes straight into COMMIT_MSG.txt.
# Nothing is standing by to replace it, so what you read is what the model said.
#
# Why </dev/null? %%bash pipes this cell into bash on STDIN and claude -p reads STDIN. Without it
# the CLI swallows every line below and they silently never run.
claude -p "$(cat /tmp/qa_demo/prompts/commit.md)$(cat /tmp/staged.diff)" \
  --model claude-sonnet-4-6 \
  --allowedTools "" \
  --max-turns 8 </dev/null > /tmp/qa_demo/COMMIT_MSG.txt

In [19]:
%%bash
# Cell: 36
# 🚦 THE GATE - a Conventional Commit subject, or nothing.
cat /tmp/qa_demo/COMMIT_MSG.txt
grep "^feat\|^fix\|^chore" /tmp/qa_demo/COMMIT_MSG.txt

feat: add phone column to dimcustomer for support callback

- Support agents had no way to call customers back; email round-trips caused escalation delays
- Phone is included in the default planner projection so it surfaces without custom queries
- Migration backfills all 16 seed rows, including international numbers, to satisfy AC #1

QA-114
feat: add phone column to dimcustomer for support callback


In [20]:
%%bash
# Cell: 37
# 👨‍💻 DEV - commit only what passed the gate.
git -C /tmp/qa_demo commit -q -F /tmp/qa_demo/COMMIT_MSG.txt
git -C /tmp/qa_demo log --oneline -2

0b5b134 feat: add phone column to dimcustomer for support callback
bc532e5 chore: baseline of the app under test, warehouse built from migrations/000


**Cell: 38**

---
## 🗺️ Step 4 · The test plan — 🧪 **QA** · `claude -p` call 2 of 7

Hat change. **From here to step 7 you are the QA Lead**, and the developer has left the room.

**At a glance**

- 🗺️ **Who does this:** 🧪 **QA** — the plan must come from what *changed* and what was *asked*, never from the author's summary of their own work.
- 🌿 **What this step does:** hands the ticket + the branch diff to a read-only reviewer and gets back a machine-checkable plan.
- 🧪 **What you should see:** `04_test_plan.json` with a `risks` list, severities and proposed test names — including at least one **critical/high** risk.
- ⚠️ **Common mistake:** asking for prose. A paragraph plan cannot stop a pipeline; a JSON `severity` field can — the gate reads the field, not the wording.

**The `claude -p` recipe — call 2 of 7**

- 🌿 **The call:** `prompts/testplan.md` + `qa_evidence/04_change.diff`, with `--add-dir /tmp/qa_demo` so it can open the repo.
- 🧪 **The grant:** `--allowedTools "Read,Grep,Glob"` — a reviewer may **look, not touch**: it cannot edit the code it judges, write the tests it specifies, or quietly fix what it finds.
- 🚦 **The gate:** `grep -c '"severity"'` then `grep -c '"severity": *"\(critical\|high\)"'` — a plan that rates a classification-spec change all-low has not read the ranger.

**A test plan is written before the tests, from the diff and the ticket.** Not from the code the
developer thinks they wrote — from what actually changed on disk, held next to what was actually
asked for. Those two documents disagree more often than anyone likes.

**The tool grant is the whole discipline of this step:**

```bash
--allowedTools "Read,Grep,Glob"
```

A reviewer may **look** and may **not touch**. It cannot edit the code it is judging, it cannot
write the tests it is specifying, and it cannot quietly fix the thing it found. Everything it
produces has to come back as text you can read, argue with, and keep.

**Why JSON and not prose.** A plan that comes back as paragraphs is a nice read and an
un-gateable artifact. A plan that comes back as JSON with a `risk` list and a `severity` field
is something `grep` can stop the pipeline on — and something you can diff against next month's
plan for the same subsystem.

**What QA is actually asking for.** Notice the prompt does not say "find bugs". It asks for the
**risk surface**: given this diff, in this application, which behaviours could now be wrong? The
difference matters — "find bugs" gets you a code review, and you already have one of those in
step 11. This step is asking *what must be proven before this ships*.

> 📌 **The governance question QA must always ask of this repo:** every column in the catalog
> either carries a classification tag or is deliberately public. So — *what did this diff add to
> the data model, and what tag did it get?* Hold that thought; do not answer it yourself. Let the
> plan and then the test say it.

**Cell: 39**

```mermaid
flowchart LR
    TICK["🎟️ QA-114.md<br/>what was asked"] --> CALL
    DIFF["📄 the diff<br/>what changed"] --> CALL["🎯 claude -p<br/>Read,Grep,Glob<br/>may look, not touch"]
    REPO["📁 /tmp/qa_demo<br/>ranger.py, catalog.py"] --> CALL
    CALL --> JSON["🗺️ test_plan.json<br/>risks + severity + test names"]
    JSON --> GATE{"🚦 has risks[]<br/>and a critical one?"}
    GATE -->|no| STOP["🛑 a plan with no risk<br/>is not a plan"]
    GATE -->|yes| NEXT["✍️ step 5 writes the cases"]
```

In [21]:
%%bash
# Cell: 40
# ============================================================================
# 🧪 QA - STEP 4: the test plan, written from the diff and the ticket
# ============================================================================

# The data half: what actually changed in this branch, against the baseline.
git -C /tmp/qa_demo diff main...HEAD > /tmp/qa_demo/qa_evidence/04_change.diff
wc -l /tmp/qa_demo/qa_evidence/04_change.diff

cat > /tmp/qa_demo/prompts/testplan.md <<'EOF'
You are a QA Lead on a data governance platform. You are writing the TEST PLAN for a change,
before any test code exists. You may read the repository but you may not change it.

The application: an agent turns English into SQL; Apache Ranger (simulated in core/ranger.py)
enforces a row-level tenant filter and column masking. The masks Ranger applies are derived from
classification tags declared in core/catalog.py - a column with no tag is, by definition, not
masked for anyone. core/guardrails.py is a second, independent scan of the output that looks for
raw sensitive values that should never have survived masking.

The data lives in a SQLite warehouse (warehouse/polaris.db) built by replaying migrations/*.sql
in order; core/data.py reads SELECT * from it. The catalog in core/catalog.py is a SEPARATE
schema - the one Ranger compiles masks from. A migration changes the warehouse; nothing changes
the catalog unless a developer does.

Read these files before answering:
  /tmp/qa_demo/tickets/QA-114.md      - what was requested
  /tmp/qa_demo/migrations/001_qa114_add_phone.sql - the schema change
  /tmp/qa_demo/core/catalog.py        - the classification spec
  /tmp/qa_demo/core/ranger.py         - the enforcement point
  /tmp/qa_demo/core/guardrails.py     - the output scanner
  /tmp/qa_demo/tests/test_flow.py     - the suite that exists today

Produce the risk surface of this change. For each risk say what could be wrong, how severe it is,
and the name of the test that would prove it. Cover at least: the three roles (admin dave,
analyst alice, support carol), the classification metadata itself, the output scanner, and
regression of the existing email/ssn/card behaviour.

Output ONLY a JSON object, no prose, no code fences:
{
  "change_summary": "one sentence",
  "risks": [
    {"id":"R1","risk":"...","severity":"critical|high|medium|low","test_name":"test_...","why_it_matters":"..."}
  ],
  "regression_areas": ["..."],
  "out_of_scope": ["..."]
}

The diff under test:

EOF


     168 /tmp/qa_demo/qa_evidence/04_change.diff


In [22]:
%%bash
# Cell: 41
cat /tmp/qa_demo/prompts/testplan.md

You are a QA Lead on a data governance platform. You are writing the TEST PLAN for a change,
before any test code exists. You may read the repository but you may not change it.

The application: an agent turns English into SQL; Apache Ranger (simulated in core/ranger.py)
enforces a row-level tenant filter and column masking. The masks Ranger applies are derived from
classification tags declared in core/catalog.py - a column with no tag is, by definition, not
masked for anyone. core/guardrails.py is a second, independent scan of the output that looks for
raw sensitive values that should never have survived masking.

The data lives in a SQLite warehouse (warehouse/polaris.db) built by replaying migrations/*.sql
in order; core/data.py reads SELECT * from it. The catalog in core/catalog.py is a SEPARATE
schema - the one Ranger compiles masks from. A migration changes the warehouse; nothing changes
the catalog unless a developer does.

Read these files before answering:
  /tmp/qa_demo/ticke

In [23]:
%%bash
# Cell: 42
# 🎯 THE CALL. A reviewer's grant: Read, Grep, Glob. It may open any file in the project and
# may not change one. --add-dir points it at a folder outside the notebook's own directory.
claude -p "$(cat /tmp/qa_demo/prompts/testplan.md)$(cat /tmp/qa_demo/qa_evidence/04_change.diff)" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Grep,Glob" \
  --add-dir /tmp/qa_demo \
  --max-turns 12 </dev/null > /tmp/qa_demo/qa_evidence/04_test_plan.json

In [24]:
%%bash
# Cell: 43
# 🧽 NORMALISE. The prompt said "output ONLY a JSON object". Some runs obey; some wrap it in a
# ```json fence, and some open with a sentence of commentary before the brace. All three are
# normal, and none of them is worth arguing about in the prompt: keep everything from the first
# { to the last } and the artifact has the same shape on every run.
#
# Prompts persuade. Gates have to hold every time - so anything a gate reads gets normalised first.
python3 -c "
import pathlib, sys
p = pathlib.Path(sys.argv[1]); s = p.read_text()
# A failed call leaves an error message, or nothing at all. Say so in one readable line
# instead of dying in a traceback - the step stops either way, but you know why.
if '{' not in s or '}' not in s:
    sys.exit(f'NO JSON in {p.name} - the call did not return an object. First 200 chars:\n{s[:200]}')
p.write_text(s[s.index('{'):s.rindex('}') + 1])" /tmp/qa_demo/qa_evidence/04_test_plan.json

In [25]:
%%bash
# Cell: 44
# 📎 EVIDENCE - the plan, pretty-printed. This is attached to the PR in step 12.
python3 -m json.tool /tmp/qa_demo/qa_evidence/04_test_plan.json

{
    "change_summary": "Adds phone column to polaris.market.dimcustomer warehouse and default planner projection, but catalog.py registers it with no classification tag, so Ranger will never mask it for any non-admin role.",
    "risks": [
        {
            "id": "R1",
            "risk": "phone column added to catalog.py with no tag (tag=None). catalog.classified_columns() returns only ['email','ssn','card']. ranger.enforce() iterates only classified_columns, so phone is never masked. alice (analyst) receives raw phone numbers in every query result.",
            "severity": "critical",
            "test_name": "test_analyst_phone_not_raw",
            "why_it_matters": "Acceptance criterion AC4 explicitly requires alice must NOT see a raw phone number. The masking framework is tag-driven; without a PII tag on phone, the enforcement point has no policy to compile and the column passes through unmasked. Phone numbers are PII under GDPR/CCPA."
        },
        {
            "id":

In [26]:
%%bash
# Cell: 45
cat /tmp/qa_demo/qa_evidence/04_test_plan.json

{
  "change_summary": "Adds phone column to polaris.market.dimcustomer warehouse and default planner projection, but catalog.py registers it with no classification tag, so Ranger will never mask it for any non-admin role.",
  "risks": [
    {
      "id": "R1",
      "risk": "phone column added to catalog.py with no tag (tag=None). catalog.classified_columns() returns only ['email','ssn','card']. ranger.enforce() iterates only classified_columns, so phone is never masked. alice (analyst) receives raw phone numbers in every query result.",
      "severity": "critical",
      "test_name": "test_analyst_phone_not_raw",
      "why_it_matters": "Acceptance criterion AC4 explicitly requires alice must NOT see a raw phone number. The masking framework is tag-driven; without a PII tag on phone, the enforcement point has no policy to compile and the column passes through unmasked. Phone numbers are PII under GDPR/CCPA."
    },
    {
      "id": "R2",
      "risk": "guardrails.py defines regex pa

In [27]:
%%bash
# Cell: 46
# 🚦 THE GATE - shape, not size. A reply is only a test plan if it carries risks with severities.
#
# Why not "is the file non-empty"? Because a CLI failure still prints text. A notebook that gated
# on emptiness once committed a message reading "Credit balance is too low".
grep -c '"severity"' /tmp/qa_demo/qa_evidence/04_test_plan.json

# And QA's real bar: a change that touches the classification spec must have produced at least one
# critical or high risk. A plan that says everything here is low has not read the ranger.
grep -c '"severity": *"\(critical\|high\)"' /tmp/qa_demo/qa_evidence/04_test_plan.json

8
4


**Cell: 47**

---
## ✍️ Step 5 · Write the test cases — 🧪 **QA** · `claude -p` call 3 of 7

**At a glance**

- 🤖 **Who does this:** 🧪 **QA** — you own the specification; the model owns the typing.
- 🧪 **What this step does:** turns the plan into one new file, `tests/test_governance.py`, with the test names dictated by QA.
- 🧪 **What you should see:** the file exists, all the named tests are present, and `core/` is untouched.
- ⚠️ **Common mistake:** letting the model choose what to test — that outsources the part of the job that was yours.

**The `claude -p` recipe — call 3 of 7**

- 🧪 **The call:** `prompts/writetests.md` (the spec, including the test names) + `qa_evidence/04_test_plan.json` (the plan), with `--add-dir /tmp/qa_demo`.
- 🧪 **The grant:** `--allowedTools "Read,Write"` + `acceptEdits` — it may create the test file and read the app to write it correctly; nothing that already exists is editable.
- 🧪 **The gate:** the file is on disk, every `def test_` name is present, and the source under test did not change — the artifact, never the exit code.

**The grant changes from `Read,Grep,Glob` to `Read,Write`.** `Write` creates new files; it does
not authorise editing existing ones. QA is allowed to add a test file and is still not allowed to
touch the code under test. That boundary is not bureaucracy — it is the only reason the RED result
in step 6 will mean anything.

**The prompt names the tests, not the assertions.** It says *which behaviours must be pinned* and
lets the agent write the code that pins them. This is the balance point of AI-assisted QA: you own
the specification, the model owns the typing. If you let the model choose what to test, you have
outsourced the part of the job that was yours.

**Eight tests, and each one exists for a reason:**

| Test | Proves |
|---|---|
| `test_phone_is_classified_in_catalog` | the **metadata** is right — this is the root-cause test |
| `test_admin_sees_raw_phone` | masking is role-scoped, not a blanket blur |
| `test_analyst_phone_is_masked` | acceptance criterion 4 |
| `test_support_phone_is_masked` | acceptance criterion 5 |
| `test_guardrails_detects_raw_phone` | the **second line of defence** actually looks for phones |
| `test_guardrails_does_not_false_alarm` | …and does not scream at a spend figure. **A constraint on the fix** |
| `test_no_leaks_for_non_admin_query` | the end-to-end audit comes back clean |
| `test_existing_masking_unchanged` | **regression** — email / ssn / card behave exactly as before |

**Test six is the one most QA suites are missing, and it was added here after a real failure:**

- 🛡️ The first time this notebook ran, the fix in step 8 added a phone-detecting regex generous enough to match `14250.75` — a customer's lifetime spend — and the audit started reporting a phone leak on a column of money. The old defect was gone; a new false alarm had replaced it.
- 🛡️ A false alarm is not a harmless bug in a security control. It is the bug that **trains people to ignore the control**, and it is invisible to every test that only asks *"did we catch the bad thing?"*
- 🔹 So QA writes the other half of the sentence: *catch the raw phone, and stay silent on everything that is not one.*
- ⚠️ Notice this test **passes in the RED run** — there is no phone detector yet, so it cannot false-alarm. It is not there to catch the defect. It is there to **constrain the fix**, which is a different and equally important job.

**Two design rules in the prompt worth stealing:**

- ✅ *Test through the public path.* The tests call `ranger.enforce` and `mcp_server.run_query` — the same doors the application uses. A test that reaches inside a private helper passes forever and proves nothing about what a user sees.
- ⚠️ *Test the EU role on EU data.* `carol` is tenant-filtered to EU, and the demo question is about a US city — so an end-to-end query returns her **zero rows**, and every assertion over zero rows is vacuously true. A test that cannot fail is worse than no test, because it reads like coverage.

**Cell: 48**

```mermaid
flowchart LR
    PLAN["🗺️ test_plan.json"] --> CALL["🎯 claude -p<br/>Read,Write<br/>may create, not edit"]
    SPEC["📄 7 test names<br/>QA owns the spec"] --> CALL
    CALL --> FILE["tests/test_governance.py"]
    FILE --> G1{"🚦 all 7 names present?"}
    G1 -->|no| STOP["🛑 incomplete suite"]
    G1 --> G2{"🚦 source untouched?"}
    G2 -->|core/ changed| STOP2["🛑 QA does not edit the code<br/>it is judging"]
    G2 -->|clean| RUN["▶️ step 6"]
```

In [28]:
%%bash
# Cell: 49
# ============================================================================
# 🧪 QA - STEP 5: generate the test cases from the plan
# ============================================================================
cat > /tmp/qa_demo/prompts/writetests.md <<'EOF'
You are a QA Lead writing an executable test suite. Write exactly one new file:

    /tmp/qa_demo/tests/test_governance.py

Rules for the file:
- Plain pytest. Module-level `def test_...` functions, plain `assert`, no classes, no fixtures,
  no mocks, no network, no Streamlit import.
- Start with this bootstrap so it runs from any working directory:
      import os, sys
      sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
- Import only from the app's public surface:
      from core import login, ranger, guardrails, mcp_server
      from core.data import load_table
      from core.planner import plan_sql
      from core import catalog
- TABLE = "polaris.market.dimcustomer"
- For per-role masking tests call ranger.enforce(principal, TABLE, load_table(TABLE)) directly.
  Do NOT use the "Top 10 Customers who live in Bollingbrook" question for the support role:
  carol is filtered to region EU and that question is about a US city, so she gets zero rows and
  every assertion over zero rows passes without proving anything. Assert len(df) > 0 first.
- A masked value is any value that is not equal to the raw value load_table() returns from the
  warehouse. Raw phone numbers there look like "630-555-0101" and "+49-89-555-0114".

Write exactly these eight tests, with these names:

1. test_phone_is_classified_in_catalog
   catalog.describe(TABLE) must list "phone" in classified_columns, and the phone Column's tag
   must be "PII". This is the root-cause test.

2. test_admin_sees_raw_phone
   dave is entitled to raw values: after ranger.enforce his phone column must still equal the raw
   seed values, and decision.masked_columns must be empty.

3. test_analyst_phone_is_masked
   alice must not see any raw phone number: no value in her phone column may match a raw seed
   phone, and "phone" must appear in decision.masked_columns.

4. test_support_phone_is_masked
   carol must not see any raw phone number either, but her mask must retain the last 4 digits so
   she can confirm identity on a call. Assert she has rows, that no value is raw, and that the
   last 4 digits of one raw EU phone survive in her masked value.

5. test_guardrails_detects_raw_phone
   Build a small pandas DataFrame containing a raw phone number, pass it to
   guardrails.scan_and_audit for alice with a ranger.RangerDecision whose masked_columns includes
   "phone", and assert "PHONE" appears in the returned audit.leaks. The second line of defence
   must be able to see a phone number at all.

6. test_guardrails_does_not_false_alarm
   The detector must be silent on values that are not sensitive. Build a small DataFrame of
   ordinary business values with NO raw phone number in it - a customer id like 1001, a plan name
   like "Gold", a lifetime spend like 14250.75, a city like "Bolingbrook" - pass it to
   guardrails.scan_and_audit for alice with a decision whose masked_columns includes "phone", and
   assert audit.leaks is empty. A security control that cries wolf on a currency amount is a
   control people learn to ignore.

7. test_no_leaks_for_non_admin_query
   Run the full mcp_server.run_query path for alice with plan_sql("Top 10 Customers who live in
   Bollingbrook") and assert audit.leaks is empty.

8. test_existing_masking_unchanged
   Regression: for alice, email / ssn / card must behave exactly as before - card fully redacted
   to "0000-0000-0000-0000", ssn not equal to the raw seed value "859-45-5012", and all three of
   email, ssn, card present in decision.masked_columns.

Write the file. Do not modify any other file. Do not run pytest. Reply with one line when done.

The test plan these cases come from:

EOF
head -20 /tmp/qa_demo/prompts/writetests.md

You are a QA Lead writing an executable test suite. Write exactly one new file:

    /tmp/qa_demo/tests/test_governance.py

Rules for the file:
- Plain pytest. Module-level `def test_...` functions, plain `assert`, no classes, no fixtures,
  no mocks, no network, no Streamlit import.
- Start with this bootstrap so it runs from any working directory:
      import os, sys
      sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
- Import only from the app's public surface:
      from core import login, ranger, guardrails, mcp_server
      from core.data import load_table
      from core.planner import plan_sql
      from core import catalog
- TABLE = "polaris.market.dimcustomer"
- For per-role masking tests call ranger.enforce(principal, TABLE, load_table(TABLE)) directly.
  Do NOT use the "Top 10 Customers who live in Bollingbrook" question for the support role:
  carol is filtered to region EU and that question is about a US city, so she gets zero rows and
 

In [29]:
%%bash
# Cell: 50
# 🎯 THE CALL. Read,Write - it may create the test file and read the app to write it correctly.
# It may not Edit: nothing that already exists is in reach.
#
# --max-turns 15: a write-a-file job measured at 10 turns, times the usual 1.5 headroom.
claude -p "$(cat /tmp/qa_demo/prompts/writetests.md)$(cat /tmp/qa_demo/qa_evidence/04_test_plan.json)" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Write" \
  --permission-mode acceptEdits \
  --add-dir /tmp/qa_demo \
  --max-turns 15 </dev/null > /tmp/qa_demo/qa_evidence/05_write_reply.txt

cat /tmp/qa_demo/qa_evidence/05_write_reply.txt

`/tmp/qa_demo/tests/test_governance.py` written — 8 tests, no fixtures/mocks/network/Streamlit.


In [30]:
%%bash
# Cell: 51
cat /tmp/qa_demo/qa_evidence/05_write_reply.txt

`/tmp/qa_demo/tests/test_governance.py` written — 8 tests, no fixtures/mocks/network/Streamlit.


In [31]:
%%bash
# Cell: 52
cat /tmp/qa_demo/tests/test_governance.py

import os, sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

import pandas as pd

from core import login, ranger, guardrails, mcp_server
from core.data import load_table
from core.planner import plan_sql
from core import catalog

TABLE = "polaris.market.dimcustomer"


# ---------------------------------------------------------------------------
# 1. Root-cause: catalog must tag phone as PII
# ---------------------------------------------------------------------------

def test_phone_is_classified_in_catalog():
    desc = catalog.describe(TABLE)
    assert "phone" in desc["classified_columns"], (
        f"'phone' missing from classified_columns; got {desc['classified_columns']}"
    )
    phone_col = catalog.CATALOG[TABLE].column("phone")
    assert phone_col is not None, "phone column not registered in catalog"
    assert phone_col.tag == "PII", f"phone.tag={phone_col.tag!r}, expected 'PII'"


# -------------------------------------------------------

In [32]:
%%bash
# Cell: 53
# 🚦 THE GATE - the artifact, not the exit code.
#
# claude exits non-zero when it runs out of turns even if the file already landed, and exits 0
# when it decides to write nothing at all. Only the file system knows which happened.
ls -l /tmp/qa_demo/tests/test_governance.py

# All eight tests, by name. A suite missing the root-cause test would go green after a fix that
# only patched the symptom.
grep -o "^def test_[a-z_]*" /tmp/qa_demo/tests/test_governance.py

# Last line of its own cell: a short suite is an incomplete specification, and step 8 is about to
# treat this file as the definition of "done".
test "$(grep -c '^def test_' /tmp/qa_demo/tests/test_governance.py)" -ge 8
%%bash
# Cell: 54
# 🚦 THE SECOND GATE - QA wrote a test; QA did not touch the code under test.
# If core/ changed in this step, the RED result in step 6 would prove nothing at all.
git -C /tmp/qa_demo status --short
git -C /tmp/qa_demo diff --quiet -- core/

-rw-r--r--@ 1 satyak  staff  6375 Aug 21 22:33 /tmp/qa_demo/tests/test_governance.py
def test_phone_is_classified_in_catalog
def test_admin_sees_raw_phone
def test_analyst_phone_is_masked
def test_support_phone_is_masked
def test_guardrails_detects_raw_phone
def test_guardrails_does_not_false_alarm
def test_no_leaks_for_non_admin_query
def test_existing_masking_unchanged


bash: line 15: fg: no job control


?? COMMIT_MSG.txt
?? prompts/
?? qa_evidence/04_change.diff
?? qa_evidence/04_test_plan.json
?? qa_evidence/05_write_reply.txt
?? tests/test_governance.py


In [ ]:
%%bash
# Cell: 55
# 📎 EVIDENCE - read the suite you are about to trust. QA reviews generated tests exactly as
# carefully as it would review generated code; a wrong test is worse than no test.
cat /tmp/qa_demo/tests/test_governance.py

**Cell: 56**

---
## 🔴 Step 6 · First run — 🧪 **QA**

**At a glance**

- 🧪 **Who does this:** 🧪 **QA** — running the new suite against the developer's unfixed code.
- ✅ **What this step does:** first run of `tests/test_governance.py`, then the legacy suite that was green in step 0.
- ⚠️ **What you should see:** **RED — about 5 failures expected here, and that is success.** A first-run green would mean the tests do not test the change, and the notebook stops.
- ⚠️ **Common mistake:** treating red as bad news. The gate literally counts failures: `grep -c "failed"`.

**This is the step the whole notebook is built around.**

The feature works. The developer demonstrated it in step 2 and was not lying. Now run the tests.

**Expect RED, and gate on RED.** If this suite goes green on its first run against unfixed code,
QA has learned that the tests do not test the change — which is a defect in the tests, and the
notebook stops. A test suite that has never been seen to fail is an unverified instrument.

**What the failures will tell you:**

- 💡 `test_phone_is_classified_in_catalog` fails because `phone` carries no tag. **The root cause.**
- ⚠️ `test_analyst_phone_is_masked` and `test_support_phone_is_masked` fail because `ranger.enforce`
  masks only `classified_columns` — and with no tag, `phone` is not in that list. **The impact.**
- 💡 `test_guardrails_detects_raw_phone` fails because `guardrails.py` scans for SSNs, cards and
  emails and has never heard of a phone number. **The second line of defence was blind too.**
- 🛡️ `test_no_leaks_for_non_admin_query` fails reporting a **CARD leak** for the analyst — and there
  is no card leak. **A false alarm**, which is its own kind of damage.

**Then run the legacy suite** — `tests/test_flow.py`, the one that was green in step 0. It fails on
that same phantom card leak, and the reason is worth the read: the raw phone number sits next to
the redacted card in the output row, and the card detector's regex matches digits and separators
straight across the gap between them.

*The run recorded here was 5 failed, 2 passed. Yours can differ by a test — the suite is generated,
not copied — but the first four failures above are the ones that matter, and they are structural.*

> 🎓 **One untagged column produced three distinct failures:** unmasked personal data, a blind
> output scanner, and a false alarm in a detector that was working correctly yesterday. This is
> what people mean when they say a governance defect is not a local defect.

**Cell: 57**

```mermaid
flowchart TD
    RUN["▶️ pytest -q<br/>tests/test_governance.py"] --> OUT["🔴 5 failed, 2 passed"]
    OUT --> F1["❌ phone not classified<br/>= ROOT CAUSE"]
    OUT --> F2["❌ analyst sees raw phone"]
    OUT --> F3["❌ support sees raw phone"]
    OUT --> F4["❌ guardrails cannot see a phone"]
    OUT --> F5["❌ audit reports a CARD leak<br/>that does not exist"]
    OUT --> P1["✅ admin raw - correct"]
    LEG["▶️ legacy tests/test_flow.py"] --> FALSE["❌ same phantom CARD leak<br/>regex matches across the gap<br/>into the raw phone beside it"]
    F1 --> GATE["🚦 RED is the expected result<br/>grep -c failed"]
    FALSE --> GATE
    GATE --> REPORT["🐞 step 7: file the defect"]
```

### 🔁 The loop you are entering: RED → defect → fix → GREEN

```mermaid
flowchart LR
    RED["🔴 6 · RED run"] --> DEF["🐞 7 · defect report<br/>one root cause"]
    DEF --> FIX["🔧 8 · DEV edits core/ only"]
    FIX --> CHK["🚦 tests/ untouched?<br/>git diff --quiet -- tests/"]
    CHK --> GRN["🟢 9 · QA re-runs both suites"]
    GRN -->|still red| DEF
    GRN -->|green| MUT["🧬 10 · mutation grades the suite"]
```

- ✅ **Why the tests are untouchable during the fix:** the tests are the *specification* of "done". The fastest route to a green build is to weaken an assertion — so the pipeline forbids the route: one `git diff --quiet -- tests/`, and any fix that edited the spec is rejected.
- ⚠️ **Why it is a loop, not a straight line:** if step 9 is still red, the work goes back through the defect report — QA does not "help the fix along", and the developer does not re-run QA's cells.


In [33]:
%%bash
# Cell: 58
# ============================================================================
# 🧪 QA - STEP 6: the first run. RED is the expected outcome.
# ============================================================================
# `|| true` because a failing suite is the POINT here - we want to read the output, not stop.
# The gate lives in the next cell, where it can be the last line.
python3 -m pytest -q /tmp/qa_demo/tests/test_governance.py > /tmp/qa_demo/qa_evidence/06_red.txt 2>&1 || true


In [34]:
%%bash
# Cell: 59
cat /tmp/qa_demo/qa_evidence/06_red.txt

/Users/satyak/.pyenv/versions/3.11.10/lib/python3.11/site-packages/pytest_asyncio/plugin.py:252: PytestDeprecationWarning: The configuration option "asyncio_default_fixture_loop_scope" is unset.
The event loop scope for asynchronous fixtures will default to the fixture caching scope. Future versions of pytest-asyncio will default the loop scope for asynchronous fixtures to function scope. Set the default fixture loop scope explicitly in order to avoid unexpected behavior in the future. Valid fixture loop scopes are: "function", "class", "module", "package", "session"

  warnings.warn(PytestDeprecationWarning(_DEFAULT_FIXTURE_LOOP_SCOPE_UNSET))
F.FFF.F.                                                                 [100%]Running teardown with pytest sessionfinish...

=================================== FAILURES ===================================
_____________________ test_phone_is_classified_in_catalog ______________________

    def test_phone_is_classified_in_catalog():
        desc

In [35]:
%%bash
# Cell: 60
# 🚦 THE GATE - and it gates on FAILURE, which is not a typo.
#
# A suite that passes on its first run against unfixed code has not been shown to test the change.
# grep -c exits non-zero when the count is zero, so a green suite stops the notebook right here.
grep -c "failed" /tmp/qa_demo/qa_evidence/06_red.txt

1


In [36]:
%%bash
# Cell: 61
# 📎 EVIDENCE - the regression the developer never ran: the suite that was green in step 0.
python3 /tmp/qa_demo/tests/test_flow.py > /tmp/qa_demo/qa_evidence/06_legacy_red.txt 2>&1 || true
tail -6 /tmp/qa_demo/qa_evidence/06_legacy_red.txt

# It fails on a CARD leak. There is no card leak. Look at what the analyst actually receives:
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core import login, mcp_server
from core.planner import plan_sql
df, dec, aud = mcp_server.run_query(login('alice'), plan_sql('Top 10 Customers who live in Bollingbrook'))
print(df[['name','ssn','card','phone']].head(3).to_string(index=False))
print()
print('masked columns :', list(dec.masked_columns))
print('audit leaks    :', aud.leaks, ' <- the card IS redacted; the detector is matching')
print('                        digits across the gap into the raw phone beside it.')"

  File "/tmp/qa_demo/tests/test_flow.py", line 67, in <module>
    main()
  File "/tmp/qa_demo/tests/test_flow.py", line 51, in main
    assert not aud_n.leaks, f"no leaks expected, got {aud_n.leaks}"
           ^^^^^^^^^^^^^^^
AssertionError: no leaks expected, got ['CARD']
        name               ssn                card        phone
  Aisha Khan 2B92EDFE70B9C924… 0000-0000-0000-0000 630-555-0105
   Hana Sato 39E94245312FC223… 0000-0000-0000-0000 630-555-0109
James Carter F1E30CD71E405E35… 0000-0000-0000-0000 630-555-0102

masked columns : ['email', 'ssn', 'card']
audit leaks    : ['CARD']  <- the card IS redacted; the detector is matching
                        digits across the gap into the raw phone beside it.


**Cell: 62**

---
## 🐞 Step 7 · The defect report — 🧪 **QA** · `claude -p` call 4 of 7

**At a glance**

- 👤 **Who does this:** 🧪 **QA** — the seat that found it files it; the developer receives a document, not a shrug.
- 🚦 **What this step does:** turns the failing output + the diff into a defect report with severity, one root cause, repro steps and evidence.
- 🚦 **What you should see:** `07_defect.json` with `"severity"` at blocker/critical and a root cause naming the missing tag.
- ⚠️ **Common mistake:** listing three symptoms as three problems. Three patches, two of them wrong — the report must name **one** root cause.

**The `claude -p` recipe — call 4 of 7**

- 🌿 **The call:** `prompts/defect.md` + `/tmp/qa_failures.txt` (the failing output and the diff, already gathered).
- 📁 **The grant:** `--allowedTools ""` — a report writer has no business reading, or fixing, the repository it is reporting on.
- 🚦 **The gate:** `grep -c '"severity"'` then `grep '"severity": *"\(blocker\|critical\)"'` — a finding without a machine-readable severity cannot stop a merge, so it is not a finding.

**A failing test is not a defect report.** The test says *what* broke. The report has to say what a
person should do about it, how badly it matters, and how anyone can see it for themselves.

**Four things every defect report in this pipeline must carry:**

| Field | Why it exists |
|---|---|
| `severity` | `blocker` stops the merge in step 11. The word has to be in a machine-readable place |
| `root_cause` | the one change that makes all the symptoms go away — not a list of symptoms |
| `steps_to_reproduce` | a person on another machine must be able to see it without asking you |
| `evidence` | the exact failing test names and the exact output. Claims that cannot be checked get argued with |

**`--allowedTools ""` again.** The failing output and the diff are already in the prompt. The agent
that writes the report has no business reading — or fixing — the repository. It is writing a
document, not opening a pull request.

> 🎯 **QA's most valuable output is not a red build. It is a report whose root cause is right.**
> Three tests failed here. If the report lists three problems, the developer applies three patches
> and two of them are wrong. If it names one missing tag, the developer applies one line.

**Cell: 63**

```mermaid
flowchart LR
    RED["🔴 pytest output"] --> CALL["🎯 claude -p<br/>--allowedTools ''<br/>writes a document"]
    LEG["🔴 legacy suite output"] --> CALL
    DIFF["📄 the diff"] --> CALL
    CALL --> J["🐞 defect.json<br/>severity · root_cause<br/>repro · evidence"]
    J --> GATE{"🚦 severity present<br/>and blocker/critical?"}
    GATE -->|no| STOP["🛑 unmergeable finding<br/>with no severity is noise"]
    GATE -->|yes| DEV["👨‍💻 step 8: one line to fix"]
```

In [37]:
%%bash
# Cell: 64
# ============================================================================
# 🧪 QA - STEP 7: file the defect
# ============================================================================
cat > /tmp/qa_demo/prompts/defect.md <<'EOF'
You are a QA Lead filing a defect against a data governance platform. You have the change diff and
the output of two failing test suites. Write the defect report.

Context you need:
- migrations/*.sql change the warehouse schema (SQLite, read by core/data.py). A migration adds
  a column to the table; it does not and cannot add a classification tag.
- core/catalog.py declares columns and classification tags (PII / PCI / TENANT). It enforces
  nothing; it is the spec.
- core/ranger.py is the enforcement point. It masks exactly the columns that catalog.py reports as
  classified. A column with no tag is masked for nobody.
- core/guardrails.py is an independent scan of the already-masked output, looking for raw
  sensitive values. It knows the shape of an SSN, a card number and an email address.

Identify the single root cause that explains every failure, not one cause per failing test.
Severity guide: blocker = personal data is visible to an identity not entitled to it.

Output ONLY a JSON object, no prose, no code fences:
{
  "defect_id": "QA-114-D1",
  "title": "one line",
  "severity": "blocker|critical|major|minor",
  "affected_identities": ["..."],
  "root_cause": "the ONE change that causes all of the symptoms",
  "symptoms": ["..."],
  "steps_to_reproduce": ["..."],
  "evidence": ["failing test names and the exact assertion output"],
  "suggested_fix": ["file: what to change"],
  "regression_risk": "what a careless fix could break"
}

The failing governance suite:

EOF

# Glue the two failure logs and the diff onto the prompt as the data half.
cat /tmp/qa_demo/qa_evidence/06_red.txt /tmp/qa_demo/qa_evidence/06_legacy_red.txt /tmp/qa_demo/qa_evidence/04_change.diff > /tmp/qa_failures.txt
wc -l /tmp/qa_failures.txt

     272 /tmp/qa_failures.txt


In [38]:
%%bash
# Cell: 65
# 🎯 THE CALL. No tools: everything it needs is in the prompt, and a report writer has no reason
# to touch the repository it is reporting on.
claude -p "$(cat /tmp/qa_demo/prompts/defect.md)$(cat /tmp/qa_failures.txt)" \
  --model claude-sonnet-4-6 \
  --allowedTools "" \
  --max-turns 8 </dev/null > /tmp/qa_demo/qa_evidence/07_defect.json

# 🧽 NORMALISE - first brace to last brace, same as step 4.
python3 -c "
import pathlib, sys
p = pathlib.Path(sys.argv[1]); s = p.read_text()
# A failed call leaves an error message, or nothing at all. Say so in one readable line
# instead of dying in a traceback - the step stops either way, but you know why.
if '{' not in s or '}' not in s:
    sys.exit(f'NO JSON in {p.name} - the call did not return an object. First 200 chars:\n{s[:200]}')
p.write_text(s[s.index('{'):s.rindex('}') + 1])" /tmp/qa_demo/qa_evidence/07_defect.json

python3 -m json.tool /tmp/qa_demo/qa_evidence/07_defect.json

{
    "defect_id": "QA-114-D1",
    "title": "phone column added to catalog.py without PII tag: raw phone numbers reach all non-admin identities and card masking breaks in mcp_server path",
    "severity": "blocker",
    "affected_identities": [
        "alice (analyst, US)",
        "carol (support, EU)",
        "all non-admin identities querying polaris.market.dimcustomer"
    ],
    "root_cause": "core/catalog.py registers Column('phone', 'string') with no classification tag. ranger.py masks exactly the columns catalog.py reports as classified; with no tag, phone is classified by nobody and masked for nobody. The same untagged column appearing in the default projection (via planner.py) exposes a latent defect in the mcp_server\u2192ranger enforcement path where the presence of an unclassified catalog column in the projected result causes card masking to be skipped, surfacing a CARD leak that guardrails then detects.",
    "symptoms": [
        "catalog.describe() returns classified

In [39]:
%%bash
# Cell: 66
# 🚦 THE GATE - a finding without a severity cannot stop a merge, so it is not a finding.
grep -c '"severity"' /tmp/qa_demo/qa_evidence/07_defect.json

# And this one must be serious: an identity saw personal data it is not entitled to.
grep '"severity": *"\(blocker\|critical\)"' /tmp/qa_demo/qa_evidence/07_defect.json

1
  "severity": "blocker",


**Cell: 67**

---
## 🔧 Step 8 · Fix what QA found — 👨‍💻 **DEV** · `claude -p` call 5 of 7

Hat change. You are the developer again, and this time you have something most developers never
get: a failing test that describes the bug precisely, and a report that names one root cause.

**At a glance**

- 💡 **Who does this:** 👨‍💻 **DEV** — fixing is the builder's job. Verifying the fix is not: that stays with QA in step 9, because the eyes that missed the tag once will miss it again.
- 🧪 **What this step does:** hands the defect report to `claude -p` on the developer's grant, then checks the *artifact* — did `core/` change, did `tests/` stay untouched.
- 🧪 **What you should see:** `tag="PII"` on the phone line in `core/catalog.py`, a phone detector in `core/guardrails.py`, and `tests/` byte-identical.
- ⚠️ **Common mistake:** trusting the exit code. Out-of-turns exits 1 with the edits already landed; "decided to change nothing" exits 0. **Ask git, not the CLI.**

**The `claude -p` recipe — call 5 of 7**

- 📄 **The call:** `prompts/fix.md` + `qa_evidence/07_defect.json`, with `--add-dir /tmp/qa_demo`.
- 👤 **The grant:** `--allowedTools "Read,Edit"` + `--permission-mode acceptEdits` — the developer seat: may change existing files, with nobody standing by to click approve. That is the only mode CI can use.
- ✅ **The gate:** `git diff --quiet -- core/` must show a change **and** `git diff --quiet -- tests/` must show none. The fastest route to green is weakening an assertion, so the tests are frozen: **they are the specification.**

**The grant is the developer seat:**

```bash
--allowedTools "Read,Edit" --permission-mode acceptEdits
```

`Edit` changes files that already exist. `acceptEdits` means nobody is standing by to click
approve — which is the only mode that works in CI, and the reason the *next* cell matters more
than this one.

**The prompt forbids three things, and each ban is a real failure mode:**

1. **Do not modify the tests.** The fastest way to a green build is to weaken the assertion. A
   gate below checks `tests/` is untouched.
2. **Do not weaken masking for any role.** "Make it pass" and "make it correct" are different
   instructions, and only one of them is worth shipping.
3. **Do not change `data.py` or `planner.py`.** The feature was right. The metadata was wrong.

**And the gate is on the artifact, never the exit code.** `claude -p` exits non-zero when it runs
out of turns *even if every edit already landed*, and exits zero when it decides to change nothing
at all. Two of those four outcomes are wrong if you read the exit code as "did the work happen".
Ask git.

**Cell: 68**

```mermaid
flowchart LR
    D["🐞 defect.json<br/>one root cause"] --> CALL["🎯 claude -p<br/>Read,Edit<br/>acceptEdits · max-turns 15"]
    T["🧪 the failing tests<br/>read, never edited"] --> CALL
    CALL --> E1["core/catalog.py<br/>tag='PII'"]
    CALL --> E2["core/guardrails.py<br/>+ phone detector"]
    E1 --> G1{"🚦 git diff -- core/<br/>did anything change?"}
    E2 --> G1
    G1 -->|empty| STOP["🛑 no edit = no fix<br/>exit code would have lied"]
    G1 --> G2{"🚦 git diff --quiet -- tests/"}
    G2 -->|tests changed| STOP2["🛑 the suite was weakened,<br/>not satisfied"]
    G2 --> OK["✅ re-run in step 9"]
```

In [ ]:
%%bash
# Cell: 69
python3 -m json.tool /tmp/qa_demo/qa_evidence/07_defect.json

In [40]:
%%bash
# Cell: 70
# ============================================================================
# 👨‍💻 DEV - STEP 8: fix the root cause
# ============================================================================
cat > /tmp/qa_demo/prompts/fix.md <<'EOF'
You are a developer fixing a blocker defect filed by QA against /tmp/qa_demo.

Read, in this order:
  /tmp/qa_demo/tests/test_governance.py   - the failing tests. They define "done".
  /tmp/qa_demo/core/catalog.py            - the classification spec
  /tmp/qa_demo/core/ranger.py             - the enforcement point
  /tmp/qa_demo/core/guardrails.py         - the independent output scanner

Make every test in tests/test_governance.py pass, and keep tests/test_flow.py passing.

Hard rules:
- Do NOT modify anything under /tmp/qa_demo/tests/. The tests are the specification.
- Do NOT weaken masking, filtering or leak detection for any role to make a test pass.
- Do NOT change migrations/, core/data.py or core/planner.py. The feature itself is correct.
- Fix the root cause. Do not special-case a column name inside ranger.py.
- Any regex you add for detecting a raw phone number must not fire on an already-masked value.
  Masked values in this app look like "XXXXX0114", "A1B2C3D4E5F6…" and "0000-0000-0000-0000".
- It must not fire on ordinary business values either. A lifetime spend of 14250.75 and a customer
  id of 1001 are not phone numbers, and a detector that reports them is a detector people will
  learn to ignore. Match the shape of a phone number, not "some digits with punctuation".

When you are done, reply with a two-line summary: the files you changed and why. Do not run pytest.
The defect report:

EOF
cat /tmp/qa_demo/prompts/fix.md

You are a developer fixing a blocker defect filed by QA against /tmp/qa_demo.

Read, in this order:
  /tmp/qa_demo/tests/test_governance.py   - the failing tests. They define "done".
  /tmp/qa_demo/core/catalog.py            - the classification spec
  /tmp/qa_demo/core/ranger.py             - the enforcement point
  /tmp/qa_demo/core/guardrails.py         - the independent output scanner

Make every test in tests/test_governance.py pass, and keep tests/test_flow.py passing.

Hard rules:
- Do NOT modify anything under /tmp/qa_demo/tests/. The tests are the specification.
- Do NOT weaken masking, filtering or leak detection for any role to make a test pass.
- Do NOT change migrations/, core/data.py or core/planner.py. The feature itself is correct.
- Fix the root cause. Do not special-case a column name inside ranger.py.
- Any regex you add for detecting a raw phone number must not fire on an already-masked value.
  Masked values in this app look like "XXXXX0114", "A1B2C3D4E5F6…" and "0

In [41]:
%%bash
# Cell: 71
# 🎯 THE CALL. Read,Edit + acceptEdits: it may change files that already exist, with nobody
# standing by to approve each one. --max-turns 15, measured: this job runs 4-6 turns cold, and a
# cap set near the measurement kills exactly the careful runs you want to keep.
claude -p "$(cat /tmp/qa_demo/prompts/fix.md)$(cat /tmp/qa_demo/qa_evidence/07_defect.json)" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Edit" \
  --permission-mode acceptEdits \
  --add-dir /tmp/qa_demo \
  --max-turns 15 </dev/null > /tmp/qa_demo/qa_evidence/08_fix_reply.txt

cat /tmp/qa_demo/qa_evidence/08_fix_reply.txt

Error: Reached max turns (15)

In [42]:
%%bash
# Cell: 72
# 🚦 THE GATE - did the source actually change? This, not the exit code, is the check.
git -C /tmp/qa_demo diff --stat -- core/

# Last line of its own cell: an empty core/ diff means no fix landed, whatever the CLI printed.
git -C /tmp/qa_demo diff --quiet -- core/ && { echo "NO EDIT - the fix did not apply"; exit 1; }
echo "core/ edited"

NO EDIT - the fix did not apply


CalledProcessError: Command 'b'# Cell: 72\n# \xf0\x9f\x9a\xa6 THE GATE - did the source actually change? This, not the exit code, is the check.\ngit -C /tmp/qa_demo diff --stat -- core/\n\n# Last line of its own cell: an empty core/ diff means no fix landed, whatever the CLI printed.\ngit -C /tmp/qa_demo diff --quiet -- core/ && { echo "NO EDIT - the fix did not apply"; exit 1; }\necho "core/ edited"\n'' returned non-zero exit status 1.

In [ ]:
%%bash
# Cell: 73
# 🚦 THE SECOND GATE - the tests are the specification, and the specification does not move.
# --quiet exits 1 the moment anything under tests/ differs, and the cell stops.
git -C /tmp/qa_demo diff --quiet -- tests/
echo "tests/ untouched - the suite was satisfied, not edited"

In [ ]:
%%bash
# Cell: 74
# 📎 EVIDENCE - the fix itself, in full. QA reads this before re-running anything.
git -C /tmp/qa_demo diff -- core/ | tee /tmp/qa_demo/qa_evidence/08_fix.diff

**Cell: 75**

---
## 🟢 Step 9 · Re-run — 🧪 **QA**

Hat change. QA verifies; the developer does not verify their own fix.

**At a glance**

- 🔹 **Who does this:** 🧪 **QA** — never the developer. A builder proving their own fix is one pair of eyes checking its own work; QA re-measures with the instruments it built *before* the fix existed.
- 🧪 **What this step does:** three separate questions, three separate cells — the new suite, the legacy suite, the three-identity behaviour.
- ✅ **What you should see:** both suites **GREEN**, and `dave` raw / `alice` hashed / `carol` last-4 in the evidence table.
- ⚠️ **Common mistake:** collapsing the three checks into one command — that is how a green build starts meaning less than it should.

**Three separate questions, three separate cells.** Collapsing them into one command is how a
green build starts meaning less than it should.

1. **Does the new suite pass?** The defect is fixed.
2. **Does the legacy suite still pass?** The fix did not break what was already working. This is
   *regression* — the reason the baseline in step 0 was recorded before anyone touched anything.
3. **Does the observable behaviour match the ticket, for all three identities?** Tests assert;
   evidence persuades. The table this cell prints is what goes in the PR, because a reviewer who
   does not want to read seven tests will read three rows.

**What "fixed" looks like here:** `dave` still sees the raw number — he is entitled to it, and a
fix that blurred it for everyone would have passed a lazier test suite and broken the product.
`alice` gets a hash. `carol` gets the last four digits, which is exactly what the ticket asked for:
enough to confirm identity on a call, not enough to dial.

**Cell: 76**

```mermaid
flowchart TD
    NEW["▶️ pytest tests/test_governance.py"] --> G1{"🚦 exit 0?"}
    G1 -->|no| BACK["🛑 back to step 8"]
    G1 --> LEG["▶️ legacy tests/test_flow.py"]
    LEG --> G2{"🚦 regression clean?"}
    G2 -->|no| BACK
    G2 --> EV["📎 EVIDENCE: 3 identities"]
    EV --> R1["dave · admin → 630-555-0101 raw ✅ entitled"]
    EV --> R2["alice · analyst → 882573…  hashed ✅"]
    EV --> R3["carol · support → XXXXX0114 last-4 ✅ can confirm, cannot dial"]
```

In [ ]:
%%bash
# Cell: 77
# ============================================================================
# 🧪 QA - STEP 9a: the defect suite must now be GREEN
# ============================================================================
# No `|| true` this time. pytest is the last line, so a single failure stops the notebook.
python3 -m pytest -q /tmp/qa_demo/tests/test_governance.py | tee /tmp/qa_demo/qa_evidence/09_green.txt
python3 -m pytest -q /tmp/qa_demo/tests/test_governance.py > /dev/null 2>&1

In [ ]:
%%bash
# Cell: 78
# 🧪 QA - STEP 9b: REGRESSION. The suite that was green in step 0 must be green again.
# A fix that trades a new bug for an old one is not a fix, and only this cell can tell you.
python3 /tmp/qa_demo/tests/test_flow.py | tee /tmp/qa_demo/qa_evidence/09_legacy_green.txt
python3 /tmp/qa_demo/tests/test_flow.py > /dev/null 2>&1

In [ ]:
%%bash
# Cell: 79
# 📎 EVIDENCE - the three-identity proof, in the form a reviewer will actually read.
python3 -c "
import sys; sys.path.insert(0, '/tmp/qa_demo')
from core import login, ranger
from core.data import load_table
T = 'polaris.market.dimcustomer'
raw = load_table(T)
print(f\"{'identity':22s} {'rows':>5s}  {'phone as seen':22s} masked columns\")
print('-' * 88)
for u in ('dave', 'alice', 'carol'):
    p = login(u)
    out, dec = ranger.enforce(p, T, raw)
    print(f'{p.label[:22]:22s} {len(out):5d}  {str(out[\"phone\"].iloc[0]):22s} {list(dec.masked_columns)}')
print()
print('dave  : entitled to raw values - unchanged, and that is the point')
print('alice : phone hashed like every other PII column she touches')
print('carol : last 4 digits survive - she can confirm identity, she cannot dial the number')
" | tee /tmp/qa_demo/qa_evidence/09_identities.txt

**Cell: 80**

---
## 🧬 Step 10 · Grade the tests — 🧪 **QA**

**At a glance**

- 🔹 **Who does this:** 🧪 **QA** — grading its own instrument before trusting it.
- 🧪 **What this step does:** `sed` removes the `PII` tag from `ssn`, the suite runs, git undoes the mutant.
- ⚠️ **What you should see:** the suite goes **RED against the mutant — failures here are the pass condition** — then green again after the `git checkout`.
- ⚠️ **Common mistake:** forgetting to stage the verified fix first. `git checkout -- <file>` restores the *index* copy; unstaged, the undo silently takes the fix with it.

A green suite proves the code passes the tests. It does not prove the tests would notice if the
code were wrong. **Mutation testing** closes that gap: break the code on purpose and confirm the
suite screams.

**The mutant:** take the `PII` tag off `ssn` — the oldest, most obviously classified column in the
application. Every mask on it disappears.

**The two outcomes:**

- ⚠️ **Suite goes red.** The tests are load-bearing. This is what you want, and the cell gates on it.
- ✅ **Suite stays green.** You have just discovered that this repository's most important guarantee
  is unprotected — and you discovered it in a controlled experiment rather than in production.

**Then `git checkout` the mutant away.** The change was deliberate, temporary and reverted in the
same step. Notice this needs no cleverness: the mutation is a `sed`, and the undo is git.

> 🎓 **Where this belongs in a real pipeline:** not on every commit — mutation runs cost real time.
> Run it when a suite is new (like this one), when a suite has never failed, and on a schedule
> against the modules that enforce your guarantees. A test suite that has never been seen to fail
> is not a test suite yet.

**Cell: 81**

```mermaid
flowchart LR
    G["🟢 suite green"] --> MUT["🧬 sed: remove tag='PII'<br/>from ssn in catalog.py"]
    MUT --> RUN["▶️ pytest"]
    RUN --> Q{"did the suite notice?"}
    Q -->|red ✅| GOOD["tests are load-bearing"]
    Q -->|green ❌| BAD["🚨 the suite does not<br/>protect masking at all"]
    GOOD --> REV["git checkout -- core/catalog.py"]
    REV --> BACK["🟢 green again<br/>repo restored"]
```

In [ ]:
%%bash
# Cell: 82
# ============================================================================
# 🧪 QA - STEP 10: mutation test. Break it on purpose.
# ============================================================================
# Stage the verified fix FIRST. The undo below is `git checkout -- <file>`, which restores the
# INDEX copy - and the index still holds the pre-fix catalog staged in step 3. Without this line
# the undo would also undo the fix, silently, and step 11's reviewer would (correctly) file a
# blocker against a branch whose evidence folder says it was fixed.
git -C /tmp/qa_demo add -- core/

# The mutant: ssn loses its PII tag. Nothing else changes.
sed -i '' 's|Column("ssn",         "string", tag="PII")|Column("ssn",         "string")|' /tmp/qa_demo/core/catalog.py
grep -n 'Column("ssn"' /tmp/qa_demo/core/catalog.py

# The suite is EXPECTED to fail here, so the run is allowed to continue to the gate below.
python3 -m pytest -q /tmp/qa_demo/tests/test_governance.py > /tmp/qa_demo/qa_evidence/10_mutant.txt 2>&1 || true
tail -12 /tmp/qa_demo/qa_evidence/10_mutant.txt

In [ ]:
%%bash
# Cell: 83
# 🚦 THE GATE - the suite must have NOTICED. Zero failures here means the tests are decorative.
grep -c "failed" /tmp/qa_demo/qa_evidence/10_mutant.txt

In [ ]:
%%bash
# Cell: 84
# 🧪 QA - undo the mutant. It was deliberate, temporary, and git is the undo button.
git -C /tmp/qa_demo checkout -- core/catalog.py
grep -n 'Column("ssn"\|Column("phone"' /tmp/qa_demo/core/catalog.py   # ssn tagged again, phone still tagged

# 🚦 THE GATE - prove the repository is back where it was: green on both suites. pytest is the
# last line, so if the undo took the fix with it, the notebook stops here instead of in step 11.
python3 /tmp/qa_demo/tests/test_flow.py > /dev/null 2>&1
python3 -m pytest -q /tmp/qa_demo/tests/test_governance.py

**Cell: 85**

---
## 🛡️ Step 11 · Security review of the final diff — 🧪 **QA** · `claude -p` call 6 of 7

**At a glance**

- ✅ **Who does this:** 🧪 **QA** — a second, adversarial read after green. The tests answered the questions QA thought to ask; this step hunts the questions nobody asked.
- 🌿 **What this step does:** hands the full branch diff to a read-only reviewer and counts blockers.
- 🚦 **What you should see:** `11_review.json` with a `verdict` and **zero** `"severity": "blocker"` findings.
- ⚠️ **Common mistake:** letting a human "read the prose and decide". The pipeline reads the field — that is the whole reason the reply is JSON.

**The `claude -p` recipe — call 6 of 7**

- 🌿 **The call:** `prompts/review.md` + `qa_evidence/11_final.diff`, with `--add-dir /tmp/qa_demo`.
- 🚦 **The grant:** `--allowedTools "Read,Grep"` — narrower than step 4: this reviewer is pointed at a known diff, so `Glob` is dropped. A grant you do not need is a grant you should not give.
- 🚦 **The gate:** `grep -c '"severity": *"blocker"'` — the count must be 0. The model supplies judgement; a plain command supplies the decision.

**Why review again, after green.** The tests answer the questions QA thought to ask in step 4. This
step goes looking for the questions nobody asked. On a governance platform the interesting ones are
always the same: did the fix create a bypass, does a mask leak through a different path, is the
classification of *every* column still defensible.

**Read-only, and a narrower grant than step 4:**

```bash
--allowedTools "Read,Grep"
```

`Glob` is dropped. This reviewer is pointed at a specific diff and specific files; it does not need
to wander the tree, and a grant you do not need is a grant you should not give.

**The gate is a count, not a judgement.** The reviewer returns findings with severities; the cell
counts blockers and stops if there are any. QA does not read the prose and decide — the pipeline
reads the field. That is the whole reason the reply is JSON.

> 🎯 **The model supplies judgement. A plain command supplies the decision.** Keep those two apart
> and you can raise or lower the merge bar without touching the prompt, and you can explain to
> anyone in the room why the build went red.

**Cell: 86**

```mermaid
flowchart LR
    FIN["📄 full branch diff<br/>main...HEAD"] --> CALL["🎯 claude -p<br/>Read,Grep · max-turns 12"]
    SRC["core/catalog.py<br/>ranger.py · guardrails.py"] --> CALL
    CALL --> J["🛡️ review.json<br/>findings[] + severity"]
    J --> COUNT["grep -c blocker"]
    COUNT --> Q{"blockers == 0?"}
    Q -->|no| BACK["🛑 back to step 8<br/>the merge does not happen"]
    Q -->|yes| SIGN["✅ QA sign-off"]
```

In [ ]:
%%bash
# Cell: 87
# ============================================================================
# 🧪 QA - STEP 11: security review of everything this branch changed
# ============================================================================
git -C /tmp/qa_demo add -A
git -C /tmp/qa_demo diff --cached > /tmp/qa_demo/qa_evidence/11_final.diff
wc -l /tmp/qa_demo/qa_evidence/11_final.diff

cat > /tmp/qa_demo/prompts/review.md <<'EOF'
You are a security reviewer on a data governance platform, reviewing a branch before it merges.
You may read the repository. You may not change it.

The trust model you are reviewing against:
- core/catalog.py is the classification spec. Ranger policies are compiled from its tags.
- core/ranger.py is the ONLY enforcement point: row-filter by tenant, column masks by role.
- core/guardrails.py is an independent scan of the output - defence in depth, not the boundary.
- The app and the agent are NOT the security boundary and must hold no masking logic.

Review the diff for:
1. any path where an identity could see a value it is not entitled to
2. masks that are reversible, or that leak enough to re-identify a person
3. a leak detector that would miss the thing it was added to catch
4. a leak detector that fires on already-masked values (a false alarm trains people to ignore it)
5. any classification tag that is now missing or wrong on ANY column in the table
6. enforcement logic that drifted out of ranger.py into the app or the agent

Output ONLY a JSON object, no prose, no code fences:
{
  "verdict": "approve|changes_requested",
  "findings": [
    {"id":"F1","severity":"blocker|major|minor|nit","file":"...","finding":"...","why":"...","fix":"..."}
  ],
  "checked_and_clean": ["..."]
}

Use severity "blocker" only for something that lets an identity see data it is not entitled to.
The diff:

EOF
head -12 /tmp/qa_demo/prompts/review.md

In [ ]:
%%bash
# Cell: 88
# 🎯 THE CALL. Read,Grep only - narrower than the test-plan reviewer, because this one is pointed
# at a known diff and does not need to search the tree.
claude -p "$(cat /tmp/qa_demo/prompts/review.md)$(cat /tmp/qa_demo/qa_evidence/11_final.diff)" \
  --model claude-sonnet-4-6 \
  --allowedTools "Read,Grep" \
  --add-dir /tmp/qa_demo \
  --max-turns 12 </dev/null > /tmp/qa_demo/qa_evidence/11_review.json

# 🧽 NORMALISE - first brace to last brace, same as steps 4 and 7.
python3 -c "
import pathlib, sys
p = pathlib.Path(sys.argv[1]); s = p.read_text()
# A failed call leaves an error message, or nothing at all. Say so in one readable line
# instead of dying in a traceback - the step stops either way, but you know why.
if '{' not in s or '}' not in s:
    sys.exit(f'NO JSON in {p.name} - the call did not return an object. First 200 chars:\n{s[:200]}')
p.write_text(s[s.index('{'):s.rindex('}') + 1])" /tmp/qa_demo/qa_evidence/11_review.json

python3 -m json.tool /tmp/qa_demo/qa_evidence/11_review.json

In [ ]:
%%bash
# Cell: 89

# 🚦 THE GATE - two lines, and the second one is the merge decision.

# First: is this a real review at all? A reply with no verdict is not one.
grep -c '"verdict"' /tmp/qa_demo/qa_evidence/11_review.json

# Then: count blockers. grep -c prints the count and exits 1 when it is zero - which is exactly
# backwards for a gate, so we invert it: finding a blocker is what must stop the run.
grep -c '"severity": *"blocker"' /tmp/qa_demo/qa_evidence/11_review.json > /tmp/blockers.txt || true
cat /tmp/blockers.txt

# Last line: "0" is the only acceptable content of that file.
grep -qx "0" /tmp/blockers.txt

**Cell: 90**

---
## 📬 Step 12 · The pull request, with the evidence attached — 🤝 **DEV + QA**

**At a glance**

- 👤 **Who does this:** 🤝 **Both** — the developer opens it; QA supplies the evidence and the sign-off. Both names on the body.
- 👤 **What this step does:** assembles the PR body from `qa_evidence/`, gates on the sign-off block, prints the `gh` commands.
- 👤 **What you should see:** `PR_BODY.md` containing `## How it was verified`, `## QA sign-off` and a `Verified by:` line.
- ⚠️ **Common mistake:** writing the body from memory. A body assembled from evidence files **cannot claim a test passed that did not run**.

**The `claude -p` recipe — call 7 of 7**

- 🗺️ **The call:** `prompts/pr.md` + `/tmp/qa_pr_material.txt` (ticket + evidence, pre-gathered).
- 🧪 **The grant:** `--allowedTools ""` — every fact it may state is already in the prompt, which is precisely how you stop a PR body from inventing a test run.
- 🚦 **The gate:** `grep -c "^## QA sign-off"` and `grep "Verified by:"` — an unsigned PR does not get opened.

**The developer opens it. QA supplies the evidence and the sign-off.** Both names are on the body,
and that is not ceremony: the PR is the artifact an auditor reads in a year, and it should say who
proved what.

**Why the body is generated from the evidence folder rather than written from memory.** Everything
in `/tmp/qa_demo/qa_evidence/` was produced by a command in this notebook — the baseline, the test
plan, the RED run, the defect, the fix diff, the GREEN run, the mutation result, the review. A PR
body assembled from those files cannot claim a test passed that did not run.

**The QA sign-off block is the part reviewers actually read.** Four lines: what was tested, what
was found, what was fixed, what evidence backs it. If any of the four is missing, the PR is not
ready — regardless of what CI says.

**`gh` commands are printed, not run**, so the notebook does not need a GitHub remote to teach the
step. Copy them into a terminal on a repo you own.

### 📎 Where the PR body comes from

Every claim in the PR traces back to a file that a command wrote:

```mermaid
flowchart LR
    A["00_baseline.txt"] --> EV["📎 qa_evidence/"]
    B["04_test_plan.json"] --> EV
    C["06_red.txt · 07_defect.json"] --> EV
    D["08_fix.diff · 09_green.txt"] --> EV
    E["10_mutant.txt · 11_review.json"] --> EV
    EV --> CALL["🎯 claude -p<br/>no tools"]
    T["🎟️ QA-114.md"] --> CALL
    CALL --> PRB["📬 PR_BODY.md<br/>+ QA sign-off"]
```


**Cell: 91**

```mermaid
flowchart LR
    EV["📎 qa_evidence/<br/>baseline · plan · RED<br/>defect · fix · GREEN<br/>mutation · review"] --> CALL["🎯 claude -p<br/>--allowedTools ''"]
    TICK["🎟️ QA-114.md"] --> CALL
    CALL --> BODY["PR_BODY.md<br/>+ QA sign-off block"]
    BODY --> GATE{"🚦 has '## QA sign-off'<br/>and 'Verified by'?"}
    GATE -->|no| STOP["🛑 an unsigned PR<br/>is not reviewable"]
    GATE -->|yes| GH["gh pr create --body-file<br/>(printed)"]
```

In [ ]:
%%bash
# Cell: 92
# ============================================================================
# 🤝 DEV + QA - STEP 12: assemble the PR body from evidence, not from memory
# ============================================================================
ls -1 /tmp/qa_demo/qa_evidence/

cat > /tmp/qa_demo/prompts/pr.md <<'EOF'
You write pull request descriptions for a data governance platform. Two people share this PR: the
developer who wrote the change and the QA Lead who tested it.

Write markdown with exactly these sections, in this order:

## What changed
Two or three sentences. Reference ticket QA-114.

## Why
The user-facing reason from the ticket.

## The defect QA found
One short paragraph: what was wrong, which identities were affected, and the single root cause.
Do not list symptoms as if they were separate bugs.

## How it was verified
A markdown table with columns: Check | Result | Evidence. One row per item, using ONLY the runs
described in the material below - baseline, new governance suite, legacy regression suite,
mutation test, security review.

## Risk and rollback
Two bullets: what could still go wrong, and how to revert.

## QA sign-off
Exactly four lines, each starting with a bold label:
**Tested:** ... / **Found:** ... / **Fixed:** ... / **Evidence:** ...
Then a final line: `Verified by: QA Lead · Ticket QA-114`

Rules: no code fences around the whole reply, no preamble, no invented results. If a number is not
in the material below, do not state it.

The material:

EOF

# The data half: the ticket plus every piece of evidence this notebook produced.
cat /tmp/qa_demo/tickets/QA-114.md \
    /tmp/qa_demo/qa_evidence/07_defect.json \
    /tmp/qa_demo/qa_evidence/09_green.txt \
    /tmp/qa_demo/qa_evidence/09_legacy_green.txt \
    /tmp/qa_demo/qa_evidence/09_identities.txt \
    /tmp/qa_demo/qa_evidence/10_mutant.txt \
    /tmp/qa_demo/qa_evidence/11_review.json > /tmp/qa_pr_material.txt
wc -l /tmp/qa_pr_material.txt

In [ ]:
%%bash
# Cell: 93
# 🎯 THE CALL. No tools: every fact it may state is already in the prompt, which is precisely
# how you stop a PR body from inventing a test run that never happened.
claude -p "$(cat /tmp/qa_demo/prompts/pr.md)$(cat /tmp/qa_pr_material.txt)" \
  --model claude-sonnet-4-6 \
  --allowedTools "" \
  --max-turns 8 </dev/null > /tmp/qa_demo/PR_BODY.md

# 🧽 NORMALISE - a fence wrapped around the whole body would render as one code block on GitHub.
sed -i '' '1{/^```/d;}' /tmp/qa_demo/PR_BODY.md
sed -i '' '${/^```$/d;}' /tmp/qa_demo/PR_BODY.md

cat /tmp/qa_demo/PR_BODY.md

In [ ]:
%%bash
# Cell: 94
# 🚦 THE GATE - an unsigned PR does not get opened.
grep -c "^## How it was verified" /tmp/qa_demo/PR_BODY.md
grep -c "^## QA sign-off" /tmp/qa_demo/PR_BODY.md
grep "Verified by:" /tmp/qa_demo/PR_BODY.md

In [ ]:
%%bash
# Cell: 95
# 🤝 Commit the evidence with the change, then print the gh commands.
# The evidence folder ships WITH the branch: a reviewer clones it and reads what QA read.
git -C /tmp/qa_demo add -A
git -C /tmp/qa_demo commit -q -m "test: governance suite for QA-114 phone column, plus QA evidence"
git -C /tmp/qa_demo log --oneline

# Printed, not run - this notebook has no remote. Copy these into a repo you own.
echo
echo "----- run these where you have a GitHub remote -----"
echo "git push -u origin feature/qa-114-phone"
echo "gh pr create --title \"$(head -1 /tmp/qa_demo/COMMIT_MSG.txt)\" --body-file /tmp/qa_demo/PR_BODY.md"

**Cell: 96**

---
## 🤝 Step 13 · Second opinion, approve, merge — 🧪 **QA Lead** (a person)

**At a glance**

- 🚦 **Who does this:** 🧪 **QA Lead — a person.** Every gate so far checked a shape; only a human decides to ship.
- 📄 **What this step does:** walks the 10-line checklist, every answer read from a file in `qa_evidence/`.
- 👤 **What you should see:** 10/10 answerable from files, then `gh pr review --approve` — a signature with a name on it.
- ⚠️ **Common mistake:** approving because CI is green. CI cannot decide that last-4 is acceptable for support and unacceptable for analysts — that call belongs to a person.

**This step is run by a human, and that is the design.** Everything up to here was a machine
checking a shape: does the reply contain a severity, did the file change, did the suite go red.
None of those is a decision to ship.

**The QA Lead's pre-merge checklist** — every line answerable from a file in `qa_evidence/`:

| # | Question | Where the answer is |
|---|---|---|
| 1 | Was the baseline green before the change? | `00_baseline.txt` |
| 1b | Did the warehouse gain the column, and did the catalog gain the tag? | `02_db_schema_*.txt`, `02_schema_*.txt`, `08_fix.diff` |
| 2 | Was the test plan written from the diff, not from the code author's summary? | `04_test_plan.json` |
| 3 | Did the suite fail *before* the fix? | `06_red.txt` |
| 4 | Does the defect name one root cause? | `07_defect.json` |
| 5 | Did the fix touch only source, never tests? | `08_fix.diff` |
| 6 | Are both suites green now? | `09_green.txt`, `09_legacy_green.txt` |
| 7 | Do all three identities behave as the ticket says? | `09_identities.txt` |
| 8 | Would the suite notice if the code broke? | `10_mutant.txt` |
| 9 | Zero blockers in the security review? | `11_review.json` |
| 10 | Is the sign-off block filled in by a person who ran the checks? | `PR_BODY.md` |

**`gh pr review --approve` is a signature.** Treat it that way — it is the one artifact in this
whole pipeline with a human name attached, and it is the one that will be read first if this
change ever causes an incident.

**Cell: 97**

```mermaid
flowchart TD
    PR["📬 PR open<br/>+ evidence folder"] --> CI["⚙️ CI runs the same gates"]
    PR --> HUM["🧪 QA Lead reads<br/>the 10-line checklist"]
    CI --> BOTH{"machine green<br/>AND human satisfied?"}
    HUM --> BOTH
    BOTH -->|no| CH["gh pr review --request-changes<br/>back to step 8"]
    BOTH -->|yes| AP["gh pr review --approve<br/>✍️ a signature, with a name on it"]
    AP --> MG["gh pr merge --squash --delete-branch"]
    MG --> MAIN["🌿 main: shippable again"]
```

In [ ]:
%%bash
# Cell: 98
# ============================================================================
# 🧪 QA LEAD - STEP 13: the pre-merge checklist, answered from files
# ============================================================================
# Every artifact the checklist refers to, with its size. A zero-byte answer is a failed check.
ls -l /tmp/qa_demo/qa_evidence/

echo
echo "----- the merge itself: run by a person, on a repo with a remote -----"
echo "gh pr view --web                                    # read the diff and the evidence"
echo "gh pr review --approve --body 'QA sign-off: 10/10 checklist, evidence attached.'"
echo "gh pr merge --squash --delete-branch"

**Cell: 99**

---
## ⚙️ Step 14 · The same gates in CI — 🤝 **DEV + QA**

**At a glance**

- ⚙️ **Who does this:** 🤝 **Both** — DEV writes the workflow file; QA defines the gates it runs.
- 🧪 **What this step does:** replays the exact cells you already ran — migrations, regression, governance suite, read-only review — on every push.
- 🚦 **What you should see:** `.github/workflows/qa.yml` whose review gate is the **same `grep`** you ran by hand.
- ⚠️ **Common mistake:** letting CI's gate drift from the notebook's. Two mechanisms means two definitions of "passing", and you will eventually meet a change that satisfies only one.

**Nothing in this workflow is new.** Every job below is a cell you already ran; CI just runs them
on somebody else's machine, on every push, without anyone remembering to.

| Job | The cell it came from |
|---|---|
| *(every job)* | step 0b and step 2a — `sqlite3 … < migrations/*.sql`, in order, before any test |
| `regression` | step 0 and step 9b — the legacy suite |
| `governance` | step 9a — the new suite |
| `review` | step 11 — `claude -p`, read-only, blockers must be 0 |

**The warehouse is rebuilt, not checked in.** `warehouse/*.db` is gitignored; each job replays
`migrations/*.sql` from scratch. If a migration cannot be replayed on a clean machine, CI is the
place to find out — not the first production deploy.

**The one thing that changes in CI: the key.** `ANTHROPIC_API_KEY` comes from repository secrets,
and the review job is guarded so a fork's pull request cannot spend your budget.

**Note the gate in the review job is the same `grep` you ran by hand.** That is the property worth
protecting: if the gate in CI is a different mechanism from the gate in your notebook, you now have
two definitions of "passing" and you will eventually meet a change that satisfies only one.

> 🎓 **Where QA's judgement still lives:** CI can run the suite, count the blockers and refuse the
> merge. It cannot decide that a mask which leaks the last four digits is acceptable for support
> and unacceptable for analysts. That call was made by a person in step 1, encoded as a test in
> step 5, and enforced by a machine ever after. **That is the whole pattern.**

**Cell: 100**

```mermaid
flowchart LR
    PUSH["git push"] --> WH["🏗️ replay migrations/*.sql"]
    WH --> R["🔁 regression<br/>legacy suite"]
    WH --> G["🧪 governance<br/>new suite"]
    PUSH --> S["🛡️ review<br/>claude -p, read-only"]
    R --> M{"all three green?"}
    G --> M
    S --> BL["grep -c blocker == 0"]
    BL --> M
    M -->|no| RED["❌ merge blocked"]
    M -->|yes| OK["✅ mergeable<br/>a human still approves"]
```

In [ ]:
%%bash
# Cell: 101
# ============================================================================
# 🤝 DEV + QA - STEP 14: the same three gates, as a workflow file
# ============================================================================
mkdir -p /tmp/qa_demo/.github/workflows
cat > /tmp/qa_demo/.github/workflows/qa.yml <<'EOF'
name: qa

on:
  pull_request:
  push:
    branches: [main]

jobs:
  regression:
    name: legacy suite (must stay green)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt pytest
      # Step 0b and step 2a: rebuild the warehouse from the migrations, in order.
      - run: mkdir -p warehouse && for f in migrations/*.sql; do sqlite3 warehouse/polaris.db < "$f"; done
      # Step 0 and step 9b, unchanged.
      - run: python3 tests/test_flow.py

  governance:
    name: governance suite (QA-114)
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: { python-version: "3.11" }
      - run: pip install -r requirements.txt pytest
      # Step 0b and step 2a: rebuild the warehouse from the migrations, in order.
      - run: mkdir -p warehouse && for f in migrations/*.sql; do sqlite3 warehouse/polaris.db < "$f"; done
      # Step 9a, unchanged.
      - run: python3 -m pytest -q tests/test_governance.py

  review:
    name: security review of the diff
    runs-on: ubuntu-latest
    # A fork's pull request must not be able to spend the key.
    if: github.event.pull_request.head.repo.full_name == github.repository
    steps:
      - uses: actions/checkout@v4
        with: { fetch-depth: 0 }
      - run: npm install -g @anthropic-ai/claude-code
      - name: the diff under review
        run: git diff origin/main...HEAD > /tmp/final.diff
      - name: claude -p, read-only
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          claude -p "$(cat prompts/review.md)$(cat /tmp/final.diff)" \
            --model claude-sonnet-4-6 \
            --allowedTools "Read,Grep" \
            --max-turns 12 </dev/null > /tmp/review.json
          cat /tmp/review.json
      - name: the gate - blockers must be zero
        run: |
          grep -c '"severity": *"blocker"' /tmp/review.json > /tmp/blockers.txt || true
          grep -qx "0" /tmp/blockers.txt
EOF

# YAML is whitespace, and a workflow that does not parse fails silently until you push it.
python3 -c "
import sys
try:
    import yaml
except ImportError:
    print('pyyaml not installed - skipping parse check'); sys.exit(0)
d = yaml.safe_load(open('/tmp/qa_demo/.github/workflows/qa.yml'))
print('jobs:', list(d['jobs']))"

git -C /tmp/qa_demo add -A
git -C /tmp/qa_demo commit -q -m "ci: run the regression, governance and review gates on every PR"
git -C /tmp/qa_demo log --oneline

**Cell: 102**

---
## 🧾 The run, as a ledger

```mermaid
flowchart TD
    S0["🧪 0 · baseline GREEN"] --> S1["👨‍💻 1 · ticket + branch"]
    S1 --> S2["👨‍💻 2 · feature: migration + 2 edits"]
    S2 --> S3["👨‍💻 3 · commit msg · call 1"]
    S3 --> S4["🧪 4 · test plan · call 2"]
    S4 --> S5["🧪 5 · 7 test cases · call 3"]
    S5 --> S6["🧪 6 · RED 🔴"]
    S6 --> S7["🧪 7 · defect · call 4"]
    S7 --> S8["👨‍💻 8 · fix · call 5"]
    S8 --> S9["🧪 9 · GREEN 🟢 + regression"]
    S9 --> S10["🧪 10 · mutation"]
    S10 --> S11["🧪 11 · security review · call 6"]
    S11 --> S12["🤝 12 · PR + sign-off · call 7"]
    S12 --> S13["🧪 13 · approve, merge"]
    S13 --> S14["🤝 14 · CI runs it all again"]
```

### 👥 Who did what

| Step | 👨‍💻 DEV | 🧪 QA | Artifact that outlives the run |
|---|---|---|---|
| 0 | | owns the clean room + warehouse + baseline | `migrations/000_*.sql`, `00_baseline.txt` |
| 1 | ticket, branch | (reads the ticket in step 4) | `tickets/QA-114.md` |
| 2 | the feature | | `migrations/001_qa114_add_phone.sql` + the diff |
| 3 | commit message | | git history |
| 4 | | the test plan | `04_test_plan.json` |
| 5 | | the test cases | `tests/test_governance.py` |
| 6 | | the RED run | `06_red.txt` |
| 7 | | the defect report | `07_defect.json` |
| 8 | the fix | | `08_fix.diff` |
| 9 | | GREEN + regression + identities | `09_*.txt` |
| 10 | | mutation grading | `10_mutant.txt` |
| 11 | | security review | `11_review.json` |
| 12 | opens the PR | signs it | `PR_BODY.md` |
| 13 | | approves and merges | the approval, with a name on it |
| 14 | writes the workflow | defines the gates | `.github/workflows/qa.yml` |

### 🎯 The seven things worth taking to your own repo

1. **QA owns the environment.** A result you cannot reproduce from a known starting state is an
   anecdote. Step 0 is not setup; it is the first measurement.
2. **The baseline is evidence, not a formality.** "It was green before" is only true if you kept
   the output that says so.
3. **Gate on RED before you gate on green.** A suite that has never failed has not been shown to
   test anything.
4. **Gate on the artifact, never the exit code.** An agent out of turns exits 1 with the work done;
   an agent that changed nothing exits 0.
5. **The tests are the specification, so the fix may not edit them.** One `git diff --quiet --
   tests/` enforces the most-violated rule in AI-assisted development.
6. **Tool grants encode the seat.** `Read,Grep,Glob` is a reviewer. `Read,Write` writes a new test
   file. `Read,Edit` is the developer. `""` writes a document. Give the seat, not the keys.
7. **The model supplies judgement; a plain command supplies the decision.** Every gate here is
   `grep`, `git` or an exit status — auditable, explainable, and unchanged whether it runs in this
   notebook or in CI.

### 🧪 Exercises

1. **Break the plan, not the code.** Delete the `phone` risk from `04_test_plan.json` and re-run
   step 5. The generated suite loses its root-cause test — and step 6 still goes red, on symptoms
   only. Then run step 8 and watch a symptom-level fix pass a weaker suite. *This is the failure
   mode of AI-assisted QA, reproduced on purpose.*
2. **Add a second untagged column.** Write `migrations/002_add_date_of_birth.sql` and declare
   `date_of_birth` in `core/catalog.py`, tagged nothing. Which of the seven tests catches it? Write the one that
   should: a test that walks *every* column in the catalog and asserts each is either classified or
   on an explicit allow-list of public columns. That test would have caught QA-114 before it was
   written.
3. **Break the mask, not the tag.** Make `_hash` in `core/ranger.py` return the first two
   characters of the raw value followed by the hash. The tag is right, the column is masked, the
   suite goes green — and the data is re-identifiable. Write the test that fails.
4. **Measure your own turn budgets.** Re-run the step 8 call with `--max-turns 40
   --output-format json`, three times, and read `num_turns` from the envelope. Set the cap to your
   observed maximum times 2.5 and compare it to the 15 used here.
5. **Move a gate into CI and try to sneak past it.** Push a branch where the fix also edits
   `tests/test_governance.py` to delete an assertion. Which job catches it? If none does, add the
   `git diff --quiet -- tests/` check to `qa.yml` — the same command from step 8.

---

*AmbariCloud · Data & AI Mastering Series · notebook 10. Companion to notebook 09 (CI/CD).
System under test: `Ranagr_DataCatalog_Iceberg`.*